# 🍅 AgriTwin-GH — Tomato Growth Stage Classifier

---

## Quick-Start Instructions

### 1. Dataset Location
**Local / VS Code:** Ensure the dataset folder exists at the path below relative to your repo root:
```
data/external/Tomato Growth Stages/     ← subfolder per growth stage class
    Stage1_Seedling/
    Stage2_Early_Vegetative/
    Stage3_Flowering_Initiation/
    Stage4_Flowering/
    Stage5_Unripe/
    Stage6_Ripe/
```
Each subfolder contains images for that stage. The **exact folder name** is used as the class label.

**Google Colab:**
```python
# Option A — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
# Then set CONFIG['repo_root'] = '/content/drive/MyDrive/AgriTwin-GH'

# Option B — Upload zip and extract
# !unzip /content/dataset.zip -d /content/AgriTwin-GH
# Then set CONFIG['repo_root'] = '/content/AgriTwin-GH'
```

### 2. CONFIG Values to Adjust
| Key | Default | When to Change |
|-----|---------|----------------|
| `repo_root` | `'.'` | Set full path in Colab |
| `backbone_name` | `'EfficientNetB3'` | Try B0 (faster), ResNet50, DenseNet121 |
| `image_size` | `(300, 300)` | Match backbone native size; lower if GPU OOM |
| `batch_size` | `16` | Lower to 8 if GPU OOM with B3 at 300px |
| `epochs_warmup` | `8` | Fewer for quick tests |
| `epochs_finetune` | `15` | Main training budget |
| `loss_type` | `'ce'` | Switch to `'focal'` for heavy class imbalance |
| `mixed_precision` | `True` | Set False if numerical issues |
| `progressive_resizing` | `True` | Set False to skip progressive warmup resolution |
| `tta` | `True` | Set False to skip test-time augmentation |
| `ignore_folders` | `[]` | Add folder names to exclude from training |

### 3. How to Run
Run all cells top-to-bottom: **Runtime → Run all** (Colab) or **Run All** (VS Code).
The two training phases (warm-up → fine-tune) run sequentially — do not skip cells.

### 4. Output Locations
```
src/agritwin_gh/models/<run_id>.keras                    ← final saved model
src/agritwin_gh/models/<run_id>_best.keras               ← best checkpoint
src/agritwin_gh/models/artifacts/<run_id>/
    label_map.json
    metrics.json
    classification_report.txt
    confusion_matrix.png
    misclassified_grid.png
    roc_curves.png
    training_history.csv
    training_history_plot.png
```
---

## Section A — Environment Setup

In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — All tunable parameters in one place.
# Modify only this cell before running the notebook.
# ─────────────────────────────────────────────────────────────────────────────
import datetime

_RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    # ── Repo / Dataset Paths ──────────────────────────────────────────────────
    # In Colab, change repo_root to the full path after mounting Drive.
    # e.g. '/content/drive/MyDrive/AgriTwin-GH' or '/content/AgriTwin-GH'
    "repo_root"         : ".",
    "data_folder"       : "data/external/Tomato Growth Stages",

    # Exact subfolder names to INCLUDE as growth-stage classes.
    # The folder name itself is used as the canonical label (no remapping).
    "include_folders"   : [
        "Stage1_Seedling",
        "Stage2_Early_Vegetative",
        "Stage3_Flowering_Initiation",
        "Stage4_Flowering",
        "Stage5_Unripe",
        "Stage6_Ripe",
    ],

    # Folders to IGNORE even if present in data_folder (e.g. corrupted sets)
    "ignore_folders"    : [],

    # ── Image / Batch ─────────────────────────────────────────────────────────
    # EfficientNetB3 native resolution = 300×300.  Reduce to 224×224 if OOM.
    "image_size"        : (300, 300),   # (H, W)
    "batch_size"        : 16,
    "num_channels"      : 3,

    # ── Split Ratios (stratified; used when no pre-split folders exist) ───────
    "val_split"         : 0.15,
    "test_split"        : 0.10,
    "random_seed"       : 42,

    # ── Backbone ──────────────────────────────────────────────────────────────
    # Supported: 'EfficientNetB3' (default, higher accuracy),
    #            'EfficientNetB0', 'ResNet50', 'MobileNetV3Large', 'DenseNet121'
    "backbone_name"     : "EfficientNetB3",

    # ── Training Schedule ─────────────────────────────────────────────────────
    "epochs_warmup"     : 8,            # Phase 1: backbone frozen
    "epochs_finetune"   : 15,           # Phase 2: top backbone layers unfrozen
    "unfreeze_top_layers" : 40,         # how many top backbone layers to unfreeze

    # ── Learning Rates ────────────────────────────────────────────────────────
    "lr_warmup"         : 1e-3,
    "lr_finetune"       : 3e-5,         # lower than disease model for B3 stability

    # ── Custom Classification Head ────────────────────────────────────────────
    "head_dropout_1"    : 0.4,
    "head_units"        : 256,
    "head_dropout_2"    : 0.3,
    "num_classes"       : 6,            # Stage1 … Stage6

    # ── Loss ──────────────────────────────────────────────────────────────────
    # 'ce'    → CategoricalCrossentropy with label_smoothing
    # 'focal' → Multiclass Focal Loss (useful for imbalanced stage counts)
    "loss_type"         : "ce",
    "label_smoothing"   : 0.1,
    "focal_alpha"       : 0.25,
    "focal_gamma"       : 2.0,

    # ── Augmentation Strength (growth-stage safe — preserve morphology/colour) ─
    # Smaller rotation / zoom than disease model to avoid confusing stage cues.
    "aug_rotation_factor"   : 0.08,    # ±8% rotation  (subtle — stage shape matters)
    "aug_zoom_factor"       : 0.10,    # ±10% zoom
    "aug_flip_horizontal"   : True,    # left-right flip (safe for stages)
    "aug_flip_vertical"     : False,   # up-down flip   (set True only if justified)
    "aug_brightness_delta"  : 0.12,    # brightness shift (mild — colour is a stage cue)
    "aug_contrast_factor"   : 0.10,    # [1-f, 1+f] contrast multiply
    "aug_hue_delta"         : 0.03,    # hue shift — very subtle (green→yellow matters)
    "aug_saturation_lower"  : 0.85,
    "aug_saturation_upper"  : 1.15,
    "aug_crop_fraction"     : 0.92,    # random crop retains ≥92% of image
    "aug_cutout_fraction"   : 0.12,    # cutout patch 12% of image dimension

    # ── Mixed Precision ───────────────────────────────────────────────────────
    "mixed_precision"   : True,        # set False if you see NaN losses

    # ── Progressive Resizing (optional accuracy booster) ──────────────────────
    # If True, warm-up runs at prog_image_size, fine-tune switches to image_size.
    "progressive_resizing" : True,
    "prog_image_size"      : (224, 224),  # smaller resolution for warm-up phase

    # ── Test-Time Augmentation (optional inference accuracy booster) ──────────
    # If True, evaluation averages tta_steps augmented forward passes per image.
    "tta"                  : True,
    "tta_steps"            : 5,           # number of augmented passes to average

    # ── Output Paths ──────────────────────────────────────────────────────────
    "models_dir"        : "src/agritwin_gh/models",
    "artifacts_dir"     : "src/agritwin_gh/models/artifacts",
    "run_id"            : f"growth_stage_{_RUN_TIMESTAMP}",
}

print("CONFIG loaded.")
print(f"Run ID     : {CONFIG['run_id']}")
print(f"Backbone   : {CONFIG['backbone_name']}  ({CONFIG['image_size'][0]}×{CONFIG['image_size'][1]})")
print(f"Loss       : {CONFIG['loss_type']}  |  Classes: {CONFIG['num_classes']}")
print(f"Prog. resize: {CONFIG['progressive_resizing']}  |  TTA: {CONFIG['tta']}")


CONFIG loaded.
Run ID     : growth_stage_20260303_015753
Backbone   : EfficientNetB3  (300×300)
Loss       : ce  |  Classes: 6
Prog. resize: True  |  TTA: True


In [44]:
# ─────────────────────────────────────────────────────────────────────────────
# A-1 | Package Check & Install (uv-first, pip fallback)
# Strategy (in order):
#   1. Skip if already importable.
#   2. Try `uv pip install` via the uv CLI (fastest, preferred).
#   3. Try `python -m uv pip install` (uv installed as Python package).
#   4. Fall back to standard `pip install` (always available).
# ─────────────────────────────────────────────────────────────────────────────
import importlib
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = {
    # import_name        : pip_package_name
    "tensorflow"         : "tensorflow",
    "sklearn"            : "scikit-learn",
    "matplotlib"         : "matplotlib",
    "numpy"              : "numpy",
    "PIL"                : "Pillow",
}

def _is_package_available(import_name: str) -> bool:
    """Return True if the package can be imported."""
    return importlib.util.find_spec(import_name) is not None

def _run_cmd(cmd: list, label: str) -> bool:
    """Run a subprocess command. Return True on success."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        stderr_snippet = (result.stderr or result.stdout or "").strip()[:400]
        print(f"    [{label}] failed (exit {result.returncode}): {stderr_snippet}")
        return False
    return True

def _install_package(pip_package_name: str) -> None:
    """Install a package using the best available installer, with fallbacks."""
    install_args = ["pip", "install", pip_package_name, "-q"]

    # 1. uv CLI available globally
    uv_cli = shutil.which("uv")
    if uv_cli:
        print(f"  → Installing via uv CLI: {uv_cli} ...")
        if _run_cmd([uv_cli] + install_args, "uv-cli"):
            print(f"    ✓ Installed with uv CLI.")
            return

    # 2. uv available as a Python module inside this env
    print(f"  → Trying python -m uv pip install ...")
    if _run_cmd([sys.executable, "-m", "uv"] + install_args, "uv-module"):
        print(f"    ✓ Installed with uv module.")
        return

    # 3. Standard pip fallback (always present)
    print(f"  → Falling back to pip install {pip_package_name} ...")
    if _run_cmd([sys.executable, "-m", "pip", "install", pip_package_name, "-q"], "pip"):
        print(f"    ✓ Installed with pip.")
        return

    print(f"  ✗ WARNING: Could not install '{pip_package_name}'. "
          f"Install it manually before proceeding.")

print("Checking required packages ...")
for import_name, pip_name in REQUIRED_PACKAGES.items():
    if _is_package_available(import_name):
        print(f"  ✓ {import_name} already available")
    else:
        print(f"  ✗ {import_name} not found — installing {pip_name} ...")
        _install_package(pip_name)

print("\nAll required packages are present.")


Checking required packages ...
  ✓ tensorflow already available
  ✓ sklearn already available
  ✓ matplotlib already available
  ✓ numpy already available
  ✓ PIL already available

All required packages are present.


In [45]:
# ─────────────────────────────────────────────────────────────────────────────
# A-2 | Core Imports
# ─────────────────────────────────────────────────────────────────────────────
import abc
import json
import os
import pathlib
import random
import warnings
from collections import Counter
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")          # non-interactive backend; safe in Colab + scripts
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import keras                   # keras 3 / TF 2.16+; falls back via tensorflow
import tensorflow as tf

warnings.filterwarnings("ignore", category=UserWarning)
print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")


TensorFlow version : 2.20.0
Keras version      : 3.13.2


In [46]:
# ─────────────────────────────────────────────────────────────────────────────
# A-3 | Deterministic Seeds, GPU Detection, Mixed Precision
# ─────────────────────────────────────────────────────────────────────────────

def setup_environment(cfg: dict) -> None:
    """Configure random seeds, GPU memory growth, and optional mixed precision."""
    seed = cfg["random_seed"]

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU(s) detected: {[g.name for g in gpus]}")
        except RuntimeError as exc:
            print(f"GPU setup warning: {exc}")
    else:
        print("No GPU detected — running on CPU (training will be slow).")

    if cfg["mixed_precision"] and gpus:
        keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled: mixed_float16")
    else:
        keras.mixed_precision.set_global_policy("float32")
        print("Mixed precision disabled: float32")

setup_environment(CONFIG)

# ── Resolve REPO_ROOT ─────────────────────────────────────────────────────────
# Priority 1 → explicit CONFIG['repo_root']  (Colab / CI)
# Priority 2 → __vsc_ipynb_file__ two levels up (VS Code)
# Priority 3 → cwd() marker walk (fallback)

def _find_repo_root_by_markers(start: pathlib.Path) -> pathlib.Path:
    """Walk up from `start` until a repo-root marker file/folder is found."""
    MARKERS = ("pyproject.toml", "setup.py", ".git")
    current = start.resolve()
    for _ in range(8):
        if any((current / m).exists() for m in MARKERS):
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    return start.resolve()   # fallback

_cfg_root = CONFIG["repo_root"]

if _cfg_root != ".":
    REPO_ROOT    = pathlib.Path(_cfg_root).resolve()
    _root_source = "CONFIG['repo_root'] (explicit)"
else:
    try:
        _nb_path  = pathlib.Path(__vsc_ipynb_file__).resolve()  # noqa: F821
        _candidate = _nb_path.parent.parent
        _MARKERS   = ("pyproject.toml", "setup.py", ".git")
        if any((_candidate / m).exists() for m in _MARKERS):
            REPO_ROOT    = _candidate
            _root_source = f"__vsc_ipynb_file__ → {_nb_path.name}"
        else:
            REPO_ROOT    = _find_repo_root_by_markers(_nb_path.parent)
            _root_source = "__vsc_ipynb_file__ (marker walk)"
    except NameError:
        REPO_ROOT    = _find_repo_root_by_markers(pathlib.Path.cwd())
        _root_source = "cwd() marker walk"

DATA_ROOT  = REPO_ROOT / CONFIG["data_folder"]
MODELS_DIR = REPO_ROOT / CONFIG["models_dir"]
RUN_DIR    = REPO_ROOT / CONFIG["artifacts_dir"] / CONFIG["run_id"]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRoot source : {_root_source}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Data root   : {DATA_ROOT}  {'✓' if DATA_ROOT.exists() else '✗ NOT FOUND — check CONFIG paths'}")
print(f"Models dir  : {MODELS_DIR}")
print(f"Run dir     : {RUN_DIR}")

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"\nDATA_ROOT not found: {DATA_ROOT}\n"
        f"Fix: set CONFIG['repo_root'] to the absolute repo path, or check CONFIG['data_folder'].\n"
        f"Expected: {REPO_ROOT / 'data/external/Tomato Growth Stages'}"
    )


No GPU detected — running on CPU (training will be slow).
Mixed precision disabled: float32

Root source : __vsc_ipynb_file__ → tomato_growth_stage_classifier_train.ipynb
Repo root   : E:\AgriTwin-GH
Data root   : E:\AgriTwin-GH\data\external\Tomato Growth Stages  ✓
Models dir  : E:\AgriTwin-GH\src\agritwin_gh\models
Run dir     : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260303_015753


## Section B — Data Pipeline (Local Only)

In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# B-1 | DatasetLoader Abstract Interface
# ─────────────────────────────────────────────────────────────────────────────

class DatasetLoader(abc.ABC):
    """
    Abstract base class for dataset loaders.

    Any concrete loader MUST implement load_file_label_pairs() which
    returns a list of (absolute_image_path_str, label_str) tuples covering
    ALL splits (train + val + test combined). The caller handles splitting.

    This interface is intentionally minimal — replacing LocalFolderLoader
    with a MinIO-backed loader in a future task requires changing one line.
    """

    @abc.abstractmethod
    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """
        Returns:
            List of (image_path: str, label: str) tuples.
            image_path — absolute path to image file (JPEG/PNG).
            label      — canonical label string (exact folder name).
        """
        ...

    @abc.abstractmethod
    def get_label_names(self) -> List[str]:
        """
        Returns:
            Sorted list of unique label strings from load_file_label_pairs.
        """
        ...

    @abc.abstractmethod
    def describe(self) -> str:
        """Human-readable description of the data source."""
        ...


In [48]:
# ─────────────────────────────────────────────────────────────────────────────
# B-2 | LocalFolderLoader — Full Implementation
#
# Layout: data_root/<StageN_Name>/<images>
# Canonical label = exact subfolder name (e.g. "Stage1_Seedling").
# Folders in ignore_folders or absent from include_folders are skipped.
# ─────────────────────────────────────────────────────────────────────────────

class LocalFolderLoader(DatasetLoader):
    """
    Loads image file paths and labels from a single root folder where each
    subdirectory represents one growth-stage class.

    The EXACT folder name is used as the canonical label — no remapping.
    This ensures label_map.json is self-documenting and human-readable.

    Args:
        data_root       : Path to the Tomato Growth Stages folder.
        include_folders : List of subfolder names to include.
        ignore_folders  : List of subfolder names to explicitly skip.
    """

    VALID_EXTENSIONS: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

    def __init__(
        self,
        data_root      : pathlib.Path,
        include_folders: List[str],
        ignore_folders : Optional[List[str]] = None,
    ) -> None:
        self.data_root       = pathlib.Path(data_root)
        self.include_folders = set(include_folders)
        self.ignore_folders  = set(ignore_folders or [])
        self._file_label_pairs: Optional[List[Tuple[str, str]]] = None

    @staticmethod
    def _collect_images_from_dir(directory: pathlib.Path) -> List[str]:
        """Recursively collect all image file paths under `directory`."""
        valid_exts = LocalFolderLoader.VALID_EXTENSIONS
        return [
            str(p.resolve())
            for p in directory.rglob("*")
            if p.is_file() and p.suffix.lower() in valid_exts
        ]

    def _scan_data_root(self) -> List[Tuple[str, str]]:
        """Scan data_root sub-folders → (absolute_path, label) pairs."""
        pairs: List[Tuple[str, str]] = []
        if not self.data_root.exists():
            raise FileNotFoundError(f"Data root not found: {self.data_root}")

        for sub in sorted(self.data_root.iterdir()):
            if not sub.is_dir():
                continue
            folder_name = sub.name

            if folder_name in self.ignore_folders:
                print(f"  [SKIP] '{folder_name}' in ignore_folders — skipping.")
                continue
            if folder_name not in self.include_folders:
                print(f"  [SKIP] '{folder_name}' not in include_folders — skipping.")
                continue

            # Label = exact folder name; no remapping required
            label  = folder_name
            images = self._collect_images_from_dir(sub)
            if not images:
                print(f"  [WARN] '{folder_name}' — no images found, skipping.")
                continue
            pairs.extend((img, label) for img in images)
            print(f"  [OK]   '{folder_name}' → label='{label}'  ({len(images)} images)")

        return pairs

    def load_file_label_pairs(self) -> List[Tuple[str, str]]:
        """Scan local folders and return all (image_path, label) pairs (cached)."""
        if self._file_label_pairs is not None:
            return self._file_label_pairs

        print("Scanning growth-stage sub-folders ...")
        all_pairs = self._scan_data_root()

        if not all_pairs:
            raise RuntimeError(
                "No images found. Check that DATA_ROOT contains actual image files "
                "and that CONFIG 'include_folders' and 'data_folder' are correct."
            )

        # Shuffle once with fixed seed for reproducibility
        rng = random.Random(CONFIG["random_seed"])
        rng.shuffle(all_pairs)
        self._file_label_pairs = all_pairs
        return self._file_label_pairs

    def get_label_names(self) -> List[str]:
        """Return sorted unique label strings (sorted by stage number naturally)."""
        pairs = self.load_file_label_pairs()
        return sorted(set(label for _, label in pairs))

    def describe(self) -> str:
        pairs  = self.load_file_label_pairs()
        counts = Counter(label for _, label in pairs)
        lines  = ["LocalFolderLoader summary:",
                  f"  Total images : {len(pairs)}"]
        for lbl in self.get_label_names():
            lines.append(f"  {lbl:<40}: {counts.get(lbl, 0):>5}")
        return "\n".join(lines)


print("LocalFolderLoader — defined.")


LocalFolderLoader — defined.


In [49]:
# ─────────────────────────────────────────────────────────────────────────────
# B-3 | Instantiate Loader + Load File-Label Pairs
# ─────────────────────────────────────────────────────────────────────────────

loader: DatasetLoader = LocalFolderLoader(
    data_root       = DATA_ROOT,
    include_folders = CONFIG["include_folders"],
    ignore_folders  = CONFIG["ignore_folders"],
)

# Load all (path, label) pairs
all_file_label_pairs: List[Tuple[str, str]] = loader.load_file_label_pairs()
label_names: List[str] = loader.get_label_names()

# Integer label encoding
label_to_idx: Dict[str, int] = {lbl: i for i, lbl in enumerate(label_names)}
idx_to_label: Dict[int, str] = {i: lbl for lbl, i in label_to_idx.items()}

print("\n" + loader.describe())
print(f"\nLabel → Index mapping:")
for lbl, idx in label_to_idx.items():
    print(f"  {idx}: {lbl}")


Scanning growth-stage sub-folders ...
  [OK]   'Stage1_Seedling' → label='Stage1_Seedling'  (87 images)
  [OK]   'Stage2_Early_Vegetative' → label='Stage2_Early_Vegetative'  (229 images)
  [OK]   'Stage3_Flowering_Initiation' → label='Stage3_Flowering_Initiation'  (217 images)
  [OK]   'Stage4_Flowering' → label='Stage4_Flowering'  (3000 images)
  [OK]   'Stage5_Unripe' → label='Stage5_Unripe'  (1324 images)
  [OK]   'Stage6_Ripe' → label='Stage6_Ripe'  (1673 images)

LocalFolderLoader summary:
  Total images : 6530
  Stage1_Seedling                         :    87
  Stage2_Early_Vegetative                 :   229
  Stage3_Flowering_Initiation             :   217
  Stage4_Flowering                        :  3000
  Stage5_Unripe                           :  1324
  Stage6_Ripe                             :  1673

Label → Index mapping:
  0: Stage1_Seedling
  1: Stage2_Early_Vegetative
  2: Stage3_Flowering_Initiation
  3: Stage4_Flowering
  4: Stage5_Unripe
  5: Stage6_Ripe


In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# B-4 | Stratified Train / Val / Test Split
# Deterministic; no dependency on pre-split folder structure.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

def stratified_three_way_split(
    file_label_pairs: List[Tuple[str, str]],
    val_fraction    : float,
    test_fraction   : float,
    seed            : int,
) -> Tuple[
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
]:
    """
    Split a flat list of (path, label) pairs into train / val / test subsets
    while preserving class proportions (stratified).

    Returns:
        train_pairs, val_pairs, test_pairs
    """
    paths  = [p for p, _ in file_label_pairs]
    labels = [l for _, l in file_label_pairs]

    # Step 1: carve off test set
    paths_tv, paths_test, labels_tv, labels_test = train_test_split(
        paths, labels,
        test_size    = test_fraction,
        stratify     = labels,
        random_state = seed,
    )

    # Step 2: carve val from remaining train+val
    val_fraction_adjusted = val_fraction / (1.0 - test_fraction)
    paths_train, paths_val, labels_train, labels_val = train_test_split(
        paths_tv, labels_tv,
        test_size    = val_fraction_adjusted,
        stratify     = labels_tv,
        random_state = seed,
    )

    train_pairs = list(zip(paths_train, labels_train))
    val_pairs   = list(zip(paths_val,   labels_val))
    test_pairs  = list(zip(paths_test,  labels_test))
    return train_pairs, val_pairs, test_pairs


train_pairs, val_pairs, test_pairs = stratified_three_way_split(
    file_label_pairs = all_file_label_pairs,
    val_fraction     = CONFIG["val_split"],
    test_fraction    = CONFIG["test_split"],
    seed             = CONFIG["random_seed"],
)

print(f"Split sizes  — train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")
print("Train class distribution:")
for lbl, cnt in sorted(Counter(l for _, l in train_pairs).items()):
    print(f"  {lbl:<40}: {cnt:>4}")


Split sizes  — train: 4897, val: 980, test: 653
Train class distribution:
  Stage1_Seedling                         :   65
  Stage2_Early_Vegetative                 :  172
  Stage3_Flowering_Initiation             :  162
  Stage4_Flowering                        : 2250
  Stage5_Unripe                           :  993
  Stage6_Ripe                             : 1255


In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# B-5 | Class Weight Computation (handles stage imbalance)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

def compute_class_weights(
    train_pairs: List[Tuple[str, str]],
    label_to_idx: Dict[str, int],
) -> Dict[int, float]:
    """Return class_weight dict keyed by integer class index."""
    train_labels_int = np.array([label_to_idx[l] for _, l in train_pairs])
    classes          = np.arange(len(label_to_idx))
    weights          = compute_class_weight(
        class_weight = "balanced",
        classes      = classes,
        y            = train_labels_int,
    )
    return {int(cls): float(w) for cls, w in zip(classes, weights)}


CLASS_WEIGHTS = compute_class_weights(train_pairs, label_to_idx)

print("Class weights (higher → rarer class, stronger up-weighting):")
for idx, weight in sorted(CLASS_WEIGHTS.items()):
    print(f"  {idx}: {idx_to_label[idx]:<40}  weight = {weight:.4f}")


Class weights (higher → rarer class, stronger up-weighting):
  0: Stage1_Seedling                           weight = 12.5564
  1: Stage2_Early_Vegetative                   weight = 4.7452
  2: Stage3_Flowering_Initiation               weight = 5.0381
  3: Stage4_Flowering                          weight = 0.3627
  4: Stage5_Unripe                             weight = 0.8219
  5: Stage6_Ripe                               weight = 0.6503


In [52]:
# ─────────────────────────────────────────────────────────────────────────────
# B-6 | tf.data Pipeline — Growth-Stage Augmentation & Preprocessing
#
# Augmentation philosophy for growth-stage classification:
#   • Growth stage is indicated by morphology (plant structure) AND colour
#     (green seedling → ripe red fruit) — both are discriminative cues.
#   • Augmentation is deliberately MILDER than the disease notebook:
#       - Rotation / zoom limited to avoid distorting plant structure.
#       - Hue shift tiny (0.03) to preserve the green→yellow→red gradient.
#       - Saturation bounded near 1.0 to avoid washing out colour cues.
#   • Vertical flip is OFF by default (configurable via CONFIG).
# ─────────────────────────────────────────────────────────────────────────────

IMG_H, IMG_W = CONFIG["image_size"]
NUM_CLASSES  = len(label_names)      # ← derived from loader, not hard-coded
CONFIG["num_classes"] = NUM_CLASSES  # update CONFIG in case dynamic

# ── Backbone-specific normalisation functions ────────────────────────────────
BACKBONE_PREPROCESS_MAP = {
    "EfficientNetB0"   : keras.applications.efficientnet.preprocess_input,
    "EfficientNetB3"   : keras.applications.efficientnet.preprocess_input,
    "ResNet50"         : keras.applications.resnet.preprocess_input,
    "MobileNetV3Large" : keras.applications.mobilenet_v3.preprocess_input,
    "DenseNet121"      : keras.applications.densenet.preprocess_input,
}

_backbone_preprocess_fn = BACKBONE_PREPROCESS_MAP.get(CONFIG["backbone_name"])
if _backbone_preprocess_fn is None:
    raise KeyError(
        f"Backbone '{CONFIG['backbone_name']}' not in BACKBONE_PREPROCESS_MAP. "
        f"Supported: {list(BACKBONE_PREPROCESS_MAP)}"
    )

# ── Instantiate Keras augmentation layers ONCE at module scope ───────────────
# (tf.Variable creation inside tf.function is forbidden; layers must live here)
_rotation_layer = keras.layers.RandomRotation(
    factor    = CONFIG["aug_rotation_factor"],
    fill_mode = "reflect",
    seed      = None,
)
_zoom_layer = keras.layers.RandomZoom(
    height_factor = (-CONFIG["aug_zoom_factor"], CONFIG["aug_zoom_factor"]),
    fill_mode     = "reflect",
    seed          = None,
)

# ── Cutout / Random Erasing ──────────────────────────────────────────────────

def apply_cutout(image: tf.Tensor, cutout_frac: float) -> tf.Tensor:
    """
    Randomly erase a square patch of (cutout_frac × min_dim) pixels,
    filling with the image mean colour. Applied with 50% probability.
    """
    if tf.random.uniform(()) > 0.5:
        return image

    h, w       = tf.shape(image)[0], tf.shape(image)[1]
    patch_size = tf.cast(
        tf.cast(tf.minimum(h, w), tf.float32) * cutout_frac, tf.int32
    )
    top  = tf.random.uniform((), 0, h - patch_size, dtype=tf.int32)
    left = tf.random.uniform((), 0, w - patch_size, dtype=tf.int32)

    mean_val = tf.reduce_mean(image)
    row_mask = tf.logical_and(tf.range(h) >= top,  tf.range(h) < (top  + patch_size))
    col_mask = tf.logical_and(tf.range(w) >= left, tf.range(w) < (left + patch_size))
    patch_mask = tf.cast(
        tf.logical_and(tf.expand_dims(row_mask, 1), tf.expand_dims(col_mask, 0)),
        image.dtype,
    )
    patch_mask = tf.expand_dims(patch_mask, -1)   # [H, W, 1]
    return image * (1.0 - patch_mask) + mean_val * patch_mask


# ── Growth-stage augmentation pipeline ──────────────────────────────────────

def augment_image(image: tf.Tensor, cfg: dict) -> tf.Tensor:
    """
    Growth-stage safe augmentation. Applies a mild sequence that preserves
    both plant morphology and colour cues (critical for stage discrimination).
    """
    image = tf.cast(image, tf.float32)

    # 1. Horizontal flip (safe — left/right symmetry of plant)
    if cfg["aug_flip_horizontal"]:
        image = tf.image.random_flip_left_right(image)

    # 2. Vertical flip (OFF by default — plant uprightness is a stage cue)
    if cfg["aug_flip_vertical"]:
        image = tf.image.random_flip_up_down(image)

    # 3. Random rotation (mild — ≤ 8% to preserve plant orientation)
    image = _rotation_layer(tf.expand_dims(image, 0), training=True)[0]

    # 4. Random zoom (mild — ≤ 10%)
    image = _zoom_layer(tf.expand_dims(image, 0), training=True)[0]

    # 5. Random brightness (mild — ≤ 12%)
    image = tf.image.random_brightness(image, max_delta=cfg["aug_brightness_delta"] * 255.0)

    # 6. Random contrast (mild — ≤ 10%)
    lower_c = 1.0 - cfg["aug_contrast_factor"]
    upper_c = 1.0 + cfg["aug_contrast_factor"]
    image = tf.image.random_contrast(image, lower=lower_c, upper=upper_c)

    # 7. Random hue — VERY subtle (0.03) to preserve green→yellow→red cues
    image = tf.image.random_hue(image / 255.0, max_delta=cfg["aug_hue_delta"]) * 255.0

    # 8. Random saturation — bounded near 1.0 to preserve colour intensity
    image = tf.image.random_saturation(
        image / 255.0,
        lower = cfg["aug_saturation_lower"],
        upper = cfg["aug_saturation_upper"],
    ) * 255.0

    # 9. Random bounded crop (retains ≥ aug_crop_fraction of each dimension)
    crop_frac   = cfg["aug_crop_fraction"]
    crop_h      = tf.cast(tf.shape(image)[0], tf.float32)
    crop_w      = tf.cast(tf.shape(image)[1], tf.float32)
    crop_size_h = tf.cast(crop_h * crop_frac, tf.int32)
    crop_size_w = tf.cast(crop_w * crop_frac, tf.int32)
    image = tf.image.random_crop(image, size=[crop_size_h, crop_size_w, 3])
    image = tf.image.resize(image, [IMG_H, IMG_W])

    # 10. Small cutout (12% patch — small enough to preserve stage cues)
    image = apply_cutout(image, cfg["aug_cutout_fraction"])

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image


# ── Core image loading ────────────────────────────────────────────────────────

def load_and_resize_image(path: str, target_h: int, target_w: int) -> tf.Tensor:
    """Read image from disk, decode, and resize to (target_h, target_w, 3)."""
    raw   = tf.io.read_file(path)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, [target_h, target_w])
    image = tf.cast(image, tf.float32)
    return image


# ── tf.data map functions ─────────────────────────────────────────────────────

def parse_and_augment(path: tf.Tensor, label_idx: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Training: load at full resolution → augment → normalise → one-hot."""
    image     = load_and_resize_image(path, IMG_H, IMG_W)
    image     = augment_image(image, CONFIG)
    image     = _backbone_preprocess_fn(image)
    label_ohe = tf.one_hot(label_idx, NUM_CLASSES)
    return image, label_ohe

def parse_only(path: tf.Tensor, label_idx: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Val / Test: load → normalise → one-hot (no augmentation)."""
    image     = load_and_resize_image(path, IMG_H, IMG_W)
    image     = _backbone_preprocess_fn(image)
    label_ohe = tf.one_hot(label_idx, NUM_CLASSES)
    return image, label_ohe

# Progressive resizing version of parse_and_augment (smaller H/W for warm-up)
_PROG_H, _PROG_W = CONFIG["prog_image_size"]

def parse_and_augment_prog(path: tf.Tensor, label_idx: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """Warm-up at prog_image_size for progressive resizing."""
    image     = load_and_resize_image(path, _PROG_H, _PROG_W)
    image     = augment_image(image, CONFIG)
    image     = _backbone_preprocess_fn(image)
    label_ohe = tf.one_hot(label_idx, NUM_CLASSES)
    return image, label_ohe


# ── Build tf.data Dataset objects ─────────────────────────────────────────────

def pairs_to_tf_dataset(
    pairs       : List[Tuple[str, str]],
    label_to_idx: Dict[str, int],
    map_fn,
    batch_size  : int,
    shuffle     : bool = False,
    seed        : int  = 42,
) -> tf.data.Dataset:
    """Convert (path, label) pairs into a batched tf.data.Dataset."""
    paths      = [p for p, _ in pairs]
    label_idxs = [label_to_idx[l] for _, l in pairs]

    ds = tf.data.Dataset.from_tensor_slices((paths, label_idxs))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(pairs), seed=seed, reshuffle_each_iteration=True)

    AUTOTUNE = tf.data.AUTOTUNE
    ds = ds.map(map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds



# ── Zero-byte / corrupt file filter ─────────────────────────────────────────
import os as _os

def _filter_valid_pairs(pairs: list, tag: str = "") -> list:
    """Remove zero-byte / missing files to prevent decode_image Input-is-empty."""
    valid, skipped = [], []
    for path_str, label in pairs:
        try:
            if _os.path.getsize(path_str) > 0:
                valid.append((path_str, label))
            else:
                skipped.append(path_str)
        except OSError:
            skipped.append(path_str)
    if skipped:
        print(f"[WARN] {tag}: skipped {len(skipped)} zero-byte / missing file(s):")
        for p in skipped[:10]:
            print(f"  x  {p}")
        if len(skipped) > 10:
            print(f"  ... and {len(skipped) - 10} more")
    else:
        print(f"  {tag}: all {len(valid)} files OK")
    return valid

print("Scanning image files for corruption / zero bytes ...")
train_pairs = _filter_valid_pairs(train_pairs, "train")
val_pairs   = _filter_valid_pairs(val_pairs,   "val")
test_pairs  = _filter_valid_pairs(test_pairs,  "test")
print(f"Final counts  train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")
# ─────────────────────────────────────────────────────────────────────────────

BATCH_SIZE = CONFIG["batch_size"]
SEED       = CONFIG["random_seed"]

train_ds = pairs_to_tf_dataset(train_pairs, label_to_idx, parse_and_augment, BATCH_SIZE, shuffle=True,  seed=SEED)
val_ds   = pairs_to_tf_dataset(val_pairs,   label_to_idx, parse_only,        BATCH_SIZE, shuffle=False, seed=SEED)
test_ds  = pairs_to_tf_dataset(test_pairs,  label_to_idx, parse_only,        BATCH_SIZE, shuffle=False, seed=SEED)

# Progressive resizing warm-up dataset (smaller resolution)
if CONFIG["progressive_resizing"] and CONFIG["prog_image_size"] != CONFIG["image_size"]:
    train_ds_prog = pairs_to_tf_dataset(
        train_pairs, label_to_idx, parse_and_augment_prog, BATCH_SIZE, shuffle=True, seed=SEED
    )
    print(f"Progressive resizing enabled: warm-up at {CONFIG['prog_image_size']}, "
          f"fine-tune at {CONFIG['image_size']}")
else:
    train_ds_prog = train_ds   # same dataset if sizes match or flag disabled
    if CONFIG["progressive_resizing"]:
        print("Progressive resizing: prog_image_size == image_size, using same dataset.")

print(f"\ntrain_ds      : {len(train_pairs)} samples, {len(train_ds)} batches at {CONFIG['image_size']}")
if train_ds_prog is not train_ds:
    print(f"train_ds_prog : {len(train_pairs)} samples, {len(train_ds_prog)} batches at {CONFIG['prog_image_size']}")
print(f"val_ds        : {len(val_pairs)} samples, {len(val_ds)} batches")
print(f"test_ds       : {len(test_pairs)} samples, {len(test_ds)} batches")


Scanning image files for corruption / zero bytes ...
[WARN] train: skipped 2 zero-byte / missing file(s):
  x  E:\AgriTwin-GH\data\external\Tomato Growth Stages\Stage3_Flowering_Initiation\1710490230456.jpg
  x  E:\AgriTwin-GH\data\external\Tomato Growth Stages\Stage3_Flowering_Initiation\1710490230470.jpg
[WARN] val: skipped 1 zero-byte / missing file(s):
  x  E:\AgriTwin-GH\data\external\Tomato Growth Stages\Stage3_Flowering_Initiation\1710490230476.jpg
[WARN] test: skipped 1 zero-byte / missing file(s):
  x  E:\AgriTwin-GH\data\external\Tomato Growth Stages\Stage3_Flowering_Initiation\1710490230430.jpg
Final counts  train: 4895, val: 979, test: 652
Progressive resizing enabled: warm-up at (224, 224), fine-tune at (300, 300)

train_ds      : 4895 samples, 306 batches at (300, 300)
train_ds_prog : 4895 samples, 306 batches at (224, 224)
val_ds        : 979 samples, 62 batches
test_ds       : 652 samples, 41 batches


## Section C — Model Architecture

In [53]:
# ─────────────────────────────────────────────────────────────────────────────
# C-1 | Backbone Factory & Full Classifier Model
# Architecture: Input → Backbone (frozen) → GAP → Dropout → Dense(relu)
#               → BatchNorm → Dropout → Dense(softmax, float32)
# ─────────────────────────────────────────────────────────────────────────────

def build_backbone(name: str, input_shape: Tuple[int, int, int]) -> keras.Model:
    """
    Instantiate a pretrained backbone (ImageNet weights) with top removed.
    All layers are frozen — Phase 1 (warm-up) trains only the custom head.

    Args:
        name        : Backbone key from BACKBONE_PREPROCESS_MAP.
        input_shape : (H, W, C) — must match CONFIG image_size.

    Returns:
        Keras model (feature extractor), all layers frozen.
    """
    common_kwargs = dict(include_top=False, weights="imagenet", input_shape=input_shape)

    backbone_constructors = {
        "EfficientNetB0"   : keras.applications.EfficientNetB0,
        "EfficientNetB3"   : keras.applications.EfficientNetB3,
        "ResNet50"         : keras.applications.ResNet50,
        "MobileNetV3Large" : keras.applications.MobileNetV3Large,
        "DenseNet121"      : keras.applications.DenseNet121,
    }

    if name not in backbone_constructors:
        raise ValueError(f"Unsupported backbone: '{name}'. Choose from {list(backbone_constructors)}.")

    backbone           = backbone_constructors[name](**common_kwargs)
    backbone.trainable = False
    print(f"Backbone '{name}' loaded — {len(backbone.layers)} layers, all frozen.")
    return backbone


def build_classifier(
    backbone_name: str,
    image_size   : Tuple[int, int],
    num_classes  : int,
    dropout_1    : float,
    dense_units  : int,
    dropout_2    : float,
) -> Tuple[keras.Model, keras.Model]:
    """
    Assemble the full classification model:
        Input → Backbone (frozen) → GAP → Dropout → Dense(relu)
             → BatchNormalization → Dropout → Dense(softmax)

    Returns:
        (model, backbone_ref) — backbone_ref used to control freezing in Phase 2.
    """
    H, W     = image_size
    inputs   = keras.Input(shape=(H, W, 3), name="image_input")
    backbone = build_backbone(backbone_name, input_shape=(H, W, 3))

    x = backbone(inputs, training=False)   # BN in inference mode during warm-up

    # Classification head
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dropout(dropout_1, name="dropout_1")(x)
    x = keras.layers.Dense(dense_units, activation="relu", name="head_dense")(x)
    x = keras.layers.BatchNormalization(name="head_bn")(x)
    x = keras.layers.Dropout(dropout_2, name="dropout_2")(x)

    # Output — float32 explicit for mixed-precision numerical stability
    outputs = keras.layers.Dense(
        num_classes,
        activation = "softmax",
        dtype      = "float32",
        name       = "predictions",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name=f"{backbone_name}_growth_stage_clf")
    return model, backbone


model, backbone = build_classifier(
    backbone_name = CONFIG["backbone_name"],
    image_size    = CONFIG["image_size"],
    num_classes   = NUM_CLASSES,
    dropout_1     = CONFIG["head_dropout_1"],
    dense_units   = CONFIG["head_units"],
    dropout_2     = CONFIG["head_dropout_2"],
)

model.summary(line_length=90)


Backbone 'EfficientNetB3' loaded — 385 layers, all frozen.


Model: "EfficientNetB3_growth_stage_clf"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)              │ (None, 300, 300, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ efficientnetb3 (Functional)           │ (None, 10, 10, 1536)         │      10,783,535 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ gap (GlobalAveragePooling2D)          │ (None, 1536)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                   │ (None, 1536)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ head_dense (Dense)                    │ (None, 256)                  │         393,472 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ head_bn (BatchNormalization)          │ (None, 256)                  │           1,024 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                   │ (None, 256)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ predictions (Dense)                   │ (None, 6)                    │           1,542 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 11,179,573 (42.65 MB)

 Trainable params: 395,526 (1.51 MB)

 Non-trainable params: 10,784,047 (41.14 MB)

In [54]:
# ─────────────────────────────────────────────────────────────────────────────
# C-2 | Loss Function Factory
# ─────────────────────────────────────────────────────────────────────────────

def build_loss_fn(loss_type: str, cfg: dict):
    """
    Return a Keras loss function based on CONFIG['loss_type'].

    'ce'    → CategoricalCrossentropy with label smoothing.
    'focal' → Multiclass Focal Loss (useful when stage counts are very unequal).
    """
    if loss_type == "ce":
        loss_fn = keras.losses.CategoricalCrossentropy(
            label_smoothing = cfg["label_smoothing"],
            from_logits     = False,
        )
        print(f"Loss: CategoricalCrossentropy (label_smoothing={cfg['label_smoothing']})")
        return loss_fn

    elif loss_type == "focal":
        alpha = cfg["focal_alpha"]
        gamma = cfg["focal_gamma"]

        def focal_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
            """
            Multiclass Focal Loss (One-vs-Rest decomposition).
            y_true: one-hot  [B, C]
            y_pred: softmax  [B, C]
            """
            y_pred = tf.cast(tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7), tf.float32)
            y_true = tf.cast(y_true, tf.float32)
            ce     = -y_true * tf.math.log(y_pred)
            p_t    = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
            focal_weight = alpha * tf.pow(1.0 - p_t, gamma)
            loss   = focal_weight * tf.reduce_sum(ce, axis=-1)
            return tf.reduce_mean(loss)

        print(f"Loss: Focal (alpha={alpha}, gamma={gamma})")
        return focal_loss

    else:
        raise ValueError(f"Unknown loss_type: '{loss_type}'. Use 'ce' or 'focal'.")


loss_fn = build_loss_fn(CONFIG["loss_type"], CONFIG)


Loss: CategoricalCrossentropy (label_smoothing=0.1)


In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# C-3 | Callbacks
# ─────────────────────────────────────────────────────────────────────────────

HISTORY_CSV_PATH = str(RUN_DIR / "training_history.csv")
CHECKPOINT_PATH  = str(MODELS_DIR / f"{CONFIG['run_id']}_best.keras")

def build_callbacks(
    phase           : str,   # 'warmup' | 'finetune'
    checkpoint_path : str,
    history_csv_path: str,
) -> List[keras.callbacks.Callback]:
    """
    Build standard callbacks for one training phase.
    CSVLogger uses append=True so both phases share one file.
    """
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor              = "val_loss",
            patience             = 6,
            restore_best_weights = True,
            verbose              = 1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor  = "val_loss",
            factor   = 0.4,
            patience = 3,
            min_lr   = 1e-7,
            verbose  = 1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath          = checkpoint_path,
            monitor           = "val_accuracy",
            save_best_only    = True,
            save_weights_only = False,
            verbose           = 1,
        ),
        keras.callbacks.CSVLogger(
            filename = history_csv_path,
            append   = True,    # append so both phases write to one CSV
        ),
    ]
    print(f"Callbacks built for phase '{phase}'.")
    return callbacks

print(f"Checkpoint path  : {CHECKPOINT_PATH}")
print(f"History CSV path : {HISTORY_CSV_PATH}")


Checkpoint path  : E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260303_015753_best.keras
History CSV path : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260303_015753\training_history.csv


In [56]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAINING BYPASS — Load saved model instead of re-training
# Run THIS cell, then SKIP D-1 and D-2, and continue from D-3 onwards.
# ─────────────────────────────────────────────────────────────────────────────
import csv as _csv

_SAVED_RUN_ID = "growth_stage_20260302_170744"  # Change this to load a different run (must exist in models_dir and artifacts_dir)

# Redirect all run-specific paths to the existing saved run
CONFIG["run_id"] = _SAVED_RUN_ID
RUN_DIR          = REPO_ROOT / CONFIG["artifacts_dir"] / _SAVED_RUN_ID
CHECKPOINT_PATH  = str(MODELS_DIR / f"{_SAVED_RUN_ID}_best.keras")
HISTORY_CSV_PATH = str(RUN_DIR / "training_history.csv")

print(f"Run ID      : {CONFIG['run_id']}")
print(f"Checkpoint  : {CHECKPOINT_PATH}")
print(f"History CSV : {HISTORY_CSV_PATH}")

assert pathlib.Path(CHECKPOINT_PATH).exists(),  f"Model not found: {CHECKPOINT_PATH}"
assert pathlib.Path(HISTORY_CSV_PATH).exists(), f"History CSV not found: {HISTORY_CSV_PATH}"
print("✓ All saved files found.")

# ── Reconstruct warmup_history / finetune_history from the saved CSV ──────────
# D-3 (history plot) and the rest of the notebook expect these objects.
_WARMUP_EPOCHS = CONFIG["epochs_warmup"]   # 8

class _MockHistory:
    """Minimal stand-in for a Keras History object."""
    def __init__(self, history_dict: dict):
        self.history = history_dict

with open(HISTORY_CSV_PATH, newline="") as _fh:
    _reader = _csv.DictReader(_fh)
    _rows   = list(_reader)

_metric_keys  = [k for k in _rows[0].keys() if k != "epoch"]
_warmup_rows  = _rows[:_WARMUP_EPOCHS]
_finetune_rows = _rows[_WARMUP_EPOCHS:]

warmup_history   = _MockHistory({k: [float(r[k]) for r in _warmup_rows   if k in r] for k in _metric_keys})
finetune_history = _MockHistory({k: [float(r[k]) for r in _finetune_rows if k in r] for k in _metric_keys})

print(f"\nHistory reconstructed — warmup: {len(_warmup_rows)} epochs, "
      f"finetune: {len(_finetune_rows)} epochs")
print("\n→ Skip D-1 and D-2 (training cells), then run from D-3 (history plot) onwards.")


Run ID      : growth_stage_20260302_170744
Checkpoint  : E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras
History CSV : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\training_history.csv
✓ All saved files found.

History reconstructed — warmup: 8 epochs, finetune: 15 epochs

→ Skip D-1 and D-2 (training cells), then run from D-3 (history plot) onwards.


## Section D — Training (Two-Phase: Warm-Up → Fine-Tune)

In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# D-1 | Phase 1 — Warm-Up Training (backbone FROZEN)
# Only the custom head is trained. Uses progressive-resizing dataset if enabled.
# ─────────────────────────────────────────────────────────────────────────────

print("═" * 60)
print("PHASE 1 — WARM-UP (backbone frozen)")
if CONFIG["progressive_resizing"] and train_ds_prog is not train_ds:
    print(f"  Progressive resizing: training at {CONFIG['prog_image_size']}")
print("═" * 60)

# Ensure backbone is frozen
backbone.trainable = False
print(f"Trainable params (warmup): {model.count_params():,}")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG["lr_warmup"]),
    loss      = loss_fn,
    metrics   = ["accuracy"],
)

warmup_callbacks = build_callbacks("warmup", CHECKPOINT_PATH, HISTORY_CSV_PATH)

# Determine which training dataset to use (progressive resizing or full)
_train_ds_phase1 = train_ds_prog if CONFIG["progressive_resizing"] else train_ds

warmup_history = model.fit(
    _train_ds_phase1,
    validation_data = val_ds,
    epochs          = CONFIG["epochs_warmup"],
    class_weight    = CLASS_WEIGHTS,
    callbacks       = warmup_callbacks,
    verbose         = 1,
)

print(f"\nWarm-up complete.  "
      f"Best val_accuracy = {max(warmup_history.history['val_accuracy']):.4f}")


════════════════════════════════════════════════════════════
PHASE 1 — WARM-UP (backbone frozen)
  Progressive resizing: training at (224, 224)
════════════════════════════════════════════════════════════
Trainable params (warmup): 11,179,573
Callbacks built for phase 'warmup'.
Epoch 1/8
306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7654 - loss: 1.1362
Epoch 1: val_accuracy improved from None to 0.92135, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras

Epoch 1: finished saving model to E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras
306/306 ━━━━━━━━━━━━━━━━━━━━ 1281s 4s/step - accuracy: 0.8435 - loss: 0.8953 - val_accuracy: 0.9213 - val_loss: 0.6524 - learning_rate: 0.0010
Epoch 2/8
306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9026 - loss: 0.6953
Epoch 2: val_accuracy improved from 0.92135 to 0.94995, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# D-2 | Phase 2 — Fine-Tuning (top N backbone layers UNFROZEN)
# Uses full-resolution train_ds regardless of progressive_resizing setting.
# ─────────────────────────────────────────────────────────────────────────────

print("═" * 60)
print("PHASE 2 — FINE-TUNING (top backbone layers unfrozen)")
print(f"  Full resolution: {CONFIG['image_size']}")
print("═" * 60)

UNFREEZE_TOP_N = CONFIG["unfreeze_top_layers"]

# Unfreeze the top N layers of the backbone
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_TOP_N]:
    layer.trainable = False

# Keep ALL BatchNorm layers in inference mode to prevent instability
for layer in backbone.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

n_trainable = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreeze : {n_trainable}")
print(f"Trainable params (finetune)     : {model.count_params():,}")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG["lr_finetune"]),
    loss      = loss_fn,
    metrics   = ["accuracy"],
)

finetune_callbacks = build_callbacks("finetune", CHECKPOINT_PATH, HISTORY_CSV_PATH)

# Always use full-resolution dataset for fine-tuning
finetune_history = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = CONFIG["epochs_finetune"],
    class_weight    = CLASS_WEIGHTS,
    callbacks       = finetune_callbacks,
    verbose         = 1,
)

print(f"\nFine-tuning complete.  "
      f"Best val_accuracy = {max(finetune_history.history['val_accuracy']):.4f}")


════════════════════════════════════════════════════════════
PHASE 2 — FINE-TUNING (top backbone layers unfrozen)
  Full resolution: (300, 300)
════════════════════════════════════════════════════════════
Trainable layers after unfreeze : 8
Trainable params (finetune)     : 11,179,573
Callbacks built for phase 'finetune'.
Epoch 1/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9499 - loss: 0.5468
Epoch 1: val_accuracy improved from None to 0.97855, saving model to E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras

Epoch 1: finished saving model to E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras
306/306 ━━━━━━━━━━━━━━━━━━━━ 1436s 5s/step - accuracy: 0.9549 - loss: 0.5501 - val_accuracy: 0.9785 - val_loss: 0.5047 - learning_rate: 3.0000e-05
Epoch 2/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9586 - loss: 0.5308
Epoch 2: val_accuracy improved from 0.97855 to 0.97957, saving model to E:\AgriTwin-GH\src\agritwin

In [57]:
# ─────────────────────────────────────────────────────────────────────────────
# D-3 | Training History Plot
# ─────────────────────────────────────────────────────────────────────────────

def merge_histories(h1, h2) -> dict:
    """Concatenate two Keras History objects into a single dict."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

merged_hist = merge_histories(warmup_history, finetune_history)
warmup_end  = len(warmup_history.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_total = range(1, len(merged_hist["loss"]) + 1)

axes[0].plot(epochs_total, merged_hist["loss"],     label="Train Loss")
axes[0].plot(epochs_total, merged_hist["val_loss"], label="Val Loss")
axes[0].axvline(warmup_end + 0.5, color="grey", linestyle="--", label="Warm-up / Fine-tune")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs_total, merged_hist["accuracy"],     label="Train Acc")
axes[1].plot(epochs_total, merged_hist["val_accuracy"], label="Val Acc")
axes[1].axvline(warmup_end + 0.5, color="grey", linestyle="--", label="Warm-up / Fine-tune")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.suptitle(f"Training History — {CONFIG['backbone_name']} — Run: {CONFIG['run_id']}")
plt.tight_layout()
HISTORY_PLOT_PATH = str(RUN_DIR / "training_history_plot.png")
plt.savefig(HISTORY_PLOT_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"History plot saved → {HISTORY_PLOT_PATH}")


History plot saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\training_history_plot.png


## Section E — Evaluation Metrics

In [58]:
# ─────────────────────────────────────────────────────────────────────────────
# E-1 | Load Best Checkpoint + Run Test-Set Inference
#       Supports optional Test-Time Augmentation (CONFIG['tta'])
# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading best checkpoint from: {CHECKPOINT_PATH}")
best_model = keras.models.load_model(CHECKPOINT_PATH)

y_true_int  = []   # integer class labels
y_pred_prob = []   # softmax probabilities [N, num_classes]


def _tta_predict_batch(
    images        : tf.Tensor,
    model         : keras.Model,
    tta_steps     : int,
) -> np.ndarray:
    """
    Average model predictions over `tta_steps` random augmentations.
    Each pass applies augment_image() independently to produce a different
    view, then softmax outputs are averaged → lower variance.

    Args:
        images    : Batch tensor [B, H, W, 3] already backbone-preprocessed.
        model     : Keras model.
        tta_steps : Number of augmented forward passes to average.

    Returns:
        Averaged probability array [B, num_classes].
    """
    # De-normalise is not possible easily; instead TTA re-applies spatial
    # augmentations on the already-loaded images (they are backbone-preprocessed
    # but spatial augmentation is still valid post-normalisation).
    accumulated = None
    for _ in range(tta_steps):
        # Re-apply small spatial augment on already-resized images
        augmented = tf.map_fn(
            lambda img: _rotation_layer(tf.expand_dims(img, 0), training=True)[0],
            images,
        )
        prob_step = model.predict_on_batch(augmented)
        if accumulated is None:
            accumulated = prob_step
        else:
            accumulated = accumulated + prob_step
    return accumulated / tta_steps


use_tta = CONFIG["tta"]
tta_steps = CONFIG["tta_steps"]

if use_tta:
    print(f"TTA enabled: averaging {tta_steps} augmented passes per batch.")
else:
    print("TTA disabled: using standard single-pass inference.")

for batch_images, batch_labels_ohe in test_ds:
    if use_tta:
        probs = _tta_predict_batch(batch_images, best_model, tta_steps)
    else:
        probs = best_model.predict_on_batch(batch_images)

    y_pred_prob.append(probs)
    y_true_int.extend(np.argmax(batch_labels_ohe.numpy(), axis=1))

y_pred_prob = np.vstack(y_pred_prob)          # [N, C]
y_true_int  = np.array(y_true_int)            # [N]
y_pred_int  = np.argmax(y_pred_prob, axis=1)  # [N]

print(f"\nTest samples evaluated : {len(y_true_int)}")
print(f"Test accuracy (raw)    : {np.mean(y_true_int == y_pred_int):.4f}")


Loading best checkpoint from: E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras
TTA enabled: averaging 5 augmented passes per batch.

Test samples evaluated : 652
Test accuracy (raw)    : 0.9862


In [59]:
# ─────────────────────────────────────────────────────────────────────────────
# E-2 | Compute All Evaluation Metrics
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix as sklearn_cm,
)


def compute_all_metrics(
    y_true     : np.ndarray,
    y_pred     : np.ndarray,
    y_pred_prob: np.ndarray,
    label_names: list[str],
) -> dict:
    """
    Compute a comprehensive set of evaluation metrics.

    Args:
        y_true      : Integer ground-truth labels [N].
        y_pred      : Integer predicted labels [N].
        y_pred_prob : Softmax probability array [N, C].
        label_names : Ordered list of class name strings.

    Returns:
        Dictionary with all scalar metrics plus per-class accuracies and
        the full classification report string.
    """
    n_classes = len(label_names)

    acc_overall = float(accuracy_score(y_true, y_pred))
    prec_macro  = float(precision_score(y_true, y_pred, average="macro",    zero_division=0))
    rec_macro   = float(recall_score   (y_true, y_pred, average="macro",    zero_division=0))
    f1_macro    = float(f1_score       (y_true, y_pred, average="macro",    zero_division=0))
    prec_wtd    = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    rec_wtd     = float(recall_score   (y_true, y_pred, average="weighted", zero_division=0))
    f1_weighted = float(f1_score       (y_true, y_pred, average="weighted", zero_division=0))

    # Per-class accuracy from confusion matrix diagonal
    cm = sklearn_cm(y_true, y_pred, labels=list(range(n_classes)))
    with np.errstate(divide="ignore", invalid="ignore"):
        per_class_acc = np.where(
            cm.sum(axis=1) > 0,
            cm.diagonal() / cm.sum(axis=1),
            0.0,
        ).tolist()

    # ROC-AUC (One-vs-Rest) — multi-class
    try:
        roc_auc = float(
            roc_auc_score(
                y_true,
                y_pred_prob,
                multi_class="ovr",
                average="macro",
                labels=list(range(n_classes)),
            )
        )
    except ValueError as exc:
        print(f"[WARN] ROC-AUC could not be computed: {exc}")
        roc_auc = float("nan")

    clf_report_str = classification_report(
        y_true,
        y_pred,
        target_names=label_names,
        zero_division=0,
    )

    metrics = {
        "accuracy"              : acc_overall,
        "precision_macro"       : prec_macro,
        "recall_macro"          : rec_macro,
        "f1_macro"              : f1_macro,
        "precision_weighted"    : prec_wtd,
        "recall_weighted"       : rec_wtd,
        "f1_weighted"           : f1_weighted,
        "roc_auc_macro_ovr"     : roc_auc,
        "per_class_accuracy"    : {label_names[i]: per_class_acc[i] for i in range(n_classes)},
        "classification_report" : clf_report_str,
    }
    return metrics


all_metrics = compute_all_metrics(y_true_int, y_pred_int, y_pred_prob, label_names)

# ── Print scalar results ────────────────────────────────────────────────────
DIVIDER = "─" * 60
print(DIVIDER)
print(f"{'EVALUATION RESULTS':^60}")
print(DIVIDER)
print(f"  Accuracy                : {all_metrics['accuracy']:.4f}")
print(f"  Precision  (macro)      : {all_metrics['precision_macro']:.4f}")
print(f"  Recall     (macro)      : {all_metrics['recall_macro']:.4f}")
print(f"  F1         (macro)      : {all_metrics['f1_macro']:.4f}")
print(f"  Precision  (weighted)   : {all_metrics['precision_weighted']:.4f}")
print(f"  Recall     (weighted)   : {all_metrics['recall_weighted']:.4f}")
print(f"  F1         (weighted)   : {all_metrics['f1_weighted']:.4f}")
print(f"  ROC-AUC    (macro OvR)  : {all_metrics['roc_auc_macro_ovr']:.4f}")
print()
print(f"{'PER-CLASS ACCURACY':^60}")
print(DIVIDER)
for cls_name, cls_acc in all_metrics["per_class_accuracy"].items():
    print(f"  {cls_name:<40s}: {cls_acc:.4f}")
print(DIVIDER)
print()
print("Classification Report:")
print(all_metrics["classification_report"])


────────────────────────────────────────────────────────────
                     EVALUATION RESULTS                     
────────────────────────────────────────────────────────────
  Accuracy                : 0.9862
  Precision  (macro)      : 0.9894
  Recall     (macro)      : 0.9908
  F1         (macro)      : 0.9899
  Precision  (weighted)   : 0.9867
  Recall     (weighted)   : 0.9862
  F1         (weighted)   : 0.9862
  ROC-AUC    (macro OvR)  : 0.9990

                     PER-CLASS ACCURACY                     
────────────────────────────────────────────────────────────
  Stage1_Seedling                         : 1.0000
  Stage2_Early_Vegetative                 : 1.0000
  Stage3_Flowering_Initiation             : 1.0000
  Stage4_Flowering                        : 1.0000
  Stage5_Unripe                           : 0.9924
  Stage6_Ripe                             : 0.9521
────────────────────────────────────────────────────────────

Classification Report:
                       

In [61]:
# ─────────────────────────────────────────────────────────────────────────────
# E-3 | Confusion Matrix (normalised + raw counts, side-by-side)
# ─────────────────────────────────────────────────────────────────────────────
import itertools


def plot_confusion_matrix(
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    label_names : list,
    save_path   : Optional[pathlib.Path] = None,
) -> None:
    """
    Plot a side-by-side pair of confusion matrices:
      - Left  : row-normalised (recall per class, 0–1)
      - Right : raw integer counts

    Args:
        y_true      : Integer ground-truth labels [N].
        y_pred      : Integer predicted labels [N].
        label_names : Ordered list of class name strings.
        save_path   : If provided, saves the figure to this path (PNG).
    """
    n = len(label_names)
    cm_raw  = sklearn_cm(y_true, y_pred, labels=list(range(n)))
    row_sum = cm_raw.sum(axis=1, keepdims=True)
    cm_norm = np.where(row_sum > 0, cm_raw / row_sum, 0.0)

    short_labels = [lbl.replace("Stage", "S") for lbl in label_names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    titles     = ["Normalised (Recall)", "Raw Counts"]
    matrices   = [cm_norm, cm_raw]
    fmt_fns    = [lambda v: f"{v:.2f}", lambda v: str(int(v))]
    cmaps      = ["Blues", "Blues"]

    for ax, title, matrix, fmt_fn, cmap in zip(axes, titles, matrices, fmt_fns, cmaps):
        im = ax.imshow(matrix, interpolation="nearest", cmap=cmap)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Predicted", fontsize=11)
        ax.set_ylabel("True", fontsize=11)
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(short_labels, rotation=40, ha="right", fontsize=9)
        ax.set_yticklabels(short_labels, fontsize=9)

        thresh = matrix.max() / 2.0
        for i, j in itertools.product(range(n), range(n)):
            color = "white" if matrix[i, j] > thresh else "black"
            ax.text(j, i, fmt_fn(matrix[i, j]),
                    ha="center", va="center", fontsize=8, color=color)

    fig.suptitle(
        f"Confusion Matrix — Tomato Growth Stages\n(run: {CONFIG['run_id']})",
        fontsize=14, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Confusion matrix saved → {save_path}")

    plt.show()
    plt.close(fig)


CM_SAVE_PATH = RUN_DIR / "confusion_matrix.png"
plot_confusion_matrix(y_true_int, y_pred_int, label_names, save_path=CM_SAVE_PATH)


Confusion matrix saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\confusion_matrix.png


In [62]:
# ─────────────────────────────────────────────────────────────────────────────
# E-4 | ROC Curves (One-vs-Rest per class)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize


def plot_roc_curves(
    y_true      : np.ndarray,
    y_pred_prob : np.ndarray,
    label_names : list,
    save_path   : Optional[pathlib.Path] = None,
) -> None:
    """
    Plot One-vs-Rest ROC curves for each class.

    Args:
        y_true      : Integer ground-truth labels [N].
        y_pred_prob : Softmax probability array [N, C].
        label_names : Ordered list of class name strings.
        save_path   : If provided, saves the figure to this path (PNG).
    """
    try:
        n_classes = len(label_names)
        y_bin = label_binarize(y_true, classes=list(range(n_classes)))

        fig, ax = plt.subplots(figsize=(9, 7))
        colors = plt.cm.tab10.colors  # type: ignore[attr-defined]

        for i, (cls_name, color) in enumerate(zip(label_names, colors)):
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
            roc_auc_cls = auc(fpr, tpr)
            short = cls_name.replace("Stage", "S")
            ax.plot(fpr, tpr, color=color, lw=2,
                    label=f"{short} (AUC = {roc_auc_cls:.3f})")

        ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random (AUC = 0.500)")
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.02])
        ax.set_xlabel("False Positive Rate", fontsize=12)
        ax.set_ylabel("True Positive Rate", fontsize=12)
        ax.set_title(
            f"ROC Curves — Tomato Growth Stages (OvR)\n"
            f"Macro AUC = {all_metrics['roc_auc_macro_ovr']:.4f}",
            fontsize=13, fontweight="bold",
        )
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        if save_path is not None:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
            print(f"ROC curves saved → {save_path}")

        plt.show()
        plt.close(fig)

    except Exception as exc:
        print(f"[WARN] ROC curve plot failed: {exc}")


ROC_SAVE_PATH = RUN_DIR / "roc_curves.png"
plot_roc_curves(y_true_int, y_pred_prob, label_names, save_path=ROC_SAVE_PATH)


ROC curves saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\roc_curves.png


In [63]:
# ─────────────────────────────────────────────────────────────────────────────
# E-5 | Misclassified Samples Grid
#       Shows the n_show highest-confidence wrong predictions
# ─────────────────────────────────────────────────────────────────────────────

def build_misclassified_grid(
    test_pairs  : list,
    y_true      : np.ndarray,
    y_pred      : np.ndarray,
    y_pred_prob : np.ndarray,
    idx_to_label: dict,
    image_size  : tuple,
    n_show      : int = 25,
    n_cols      : int = 5,
    save_path   : Optional[pathlib.Path] = None,
) -> None:
    """
    Display a grid of misclassified test images sorted by wrong-prediction
    confidence (descending — hardest / most-confident mistakes first).

    Annotations (in red):
      Top    : True label
      Bottom : Predicted label + confidence %

    Args:
        test_pairs   : List of (image_path_str, true_int_label) pairs.
        y_true       : Integer ground-truth labels [N].
        y_pred       : Integer predicted labels [N].
        y_pred_prob  : Softmax probability array [N, C].
        idx_to_label : Mapping from integer index → class name string.
        image_size   : (H, W) used to resize images for display.
        n_show       : Maximum number of images to show in the grid.
        n_cols       : Number of columns in the grid.
        save_path    : If provided, saves the figure to this path (PNG).
    """
    # Identify misclassified indices
    wrong_mask = y_true != y_pred
    wrong_idxs = np.where(wrong_mask)[0]

    if len(wrong_idxs) == 0:
        print("[INFO] No misclassified samples found — perfect test accuracy!")
        return

    # Sort by confidence in the wrong predicted class (descending)
    wrong_conf = y_pred_prob[wrong_idxs, y_pred[wrong_idxs]]
    sorted_order = wrong_idxs[np.argsort(wrong_conf)[::-1]]
    show_idxs = sorted_order[:n_show]

    n_show_actual = len(show_idxs)
    n_rows = math.ceil(n_show_actual / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.5))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

    for plot_i, sample_idx in enumerate(show_idxs):
        img_path_str, _ = test_pairs[sample_idx]
        true_lbl  = idx_to_label[int(y_true[sample_idx])]
        pred_lbl  = idx_to_label[int(y_pred[sample_idx])]
        pred_conf = float(y_pred_prob[sample_idx, y_pred[sample_idx]])

        try:
            img = Image.open(img_path_str).convert("RGB").resize(image_size[::-1])
            axes[plot_i].imshow(np.array(img))
        except Exception:
            axes[plot_i].imshow(np.zeros((*image_size, 3), dtype=np.uint8))

        short_true = true_lbl.replace("Stage", "S")
        short_pred = pred_lbl.replace("Stage", "S")
        axes[plot_i].set_title(
            f"True: {short_true}",
            fontsize=8, color="red", fontweight="bold", pad=2,
        )
        axes[plot_i].set_xlabel(
            f"Pred: {short_pred}\n({pred_conf:.1%})",
            fontsize=7.5, color="red",
        )
        axes[plot_i].set_xticks([])
        axes[plot_i].set_yticks([])

    # Hide unused subplots
    for ax in axes[n_show_actual:]:
        ax.axis("off")

    fig.suptitle(
        f"Top-{n_show_actual} Misclassified Samples (sorted by confidence)\n"
        f"Total wrong: {len(wrong_idxs)} / {len(y_true)}  "
        f"({len(wrong_idxs)/len(y_true):.1%} error rate)",
        fontsize=12, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Misclassified grid saved → {save_path}")

    plt.show()
    plt.close(fig)


import math

MISCLASSIFIED_SAVE_PATH = RUN_DIR / "misclassified_grid.png"

# Reconstruct test_pairs in the same order as test_ds was built from
# (loader.test_pairs was stored in B-4)
build_misclassified_grid(
    test_pairs   = test_pairs,
    y_true       = y_true_int,
    y_pred       = y_pred_int,
    y_pred_prob  = y_pred_prob,
    idx_to_label = idx_to_label,
    image_size   = tuple(CONFIG["image_size"]),
    n_show       = 25,
    n_cols       = 5,
    save_path    = MISCLASSIFIED_SAVE_PATH,
)


Misclassified grid saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\misclassified_grid.png


## Section F — Save All Artifacts

Serialises the final model and all evaluation artifacts to disk:

| Artifact | Path |
|---|---|
| Final model (`.keras`) | `MODELS_DIR/<run_id>.keras` |
| Best checkpoint (`_best.keras`) | `MODELS_DIR/<run_id>_best.keras` |
| Label map (`label_map.json`) | `RUN_DIR/label_map.json` |
| Metrics (`metrics.json`) | `RUN_DIR/metrics.json` |
| Classification report (`.txt`) | `RUN_DIR/classification_report.txt` |
| Confusion matrix (`.png`) | `RUN_DIR/confusion_matrix.png` |
| Misclassified grid (`.png`) | `RUN_DIR/misclassified_grid.png` |
| ROC curves (`.png`) | `RUN_DIR/roc_curves.png` |
| Training history CSV | `RUN_DIR/training_history.csv` |
| Training history plot | `RUN_DIR/training_history_plot.png` |


In [64]:
# ─────────────────────────────────────────────────────────────────────────────
# F-1 | Save Final Model (.keras)
# ─────────────────────────────────────────────────────────────────────────────

FINAL_MODEL_PATH = MODELS_DIR / f"{CONFIG['run_id']}.keras"

print(f"Saving final model → {FINAL_MODEL_PATH}")
best_model.save(str(FINAL_MODEL_PATH))
print("Model saved successfully.")


Saving final model → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744.keras
Model saved successfully.


In [66]:
# ─────────────────────────────────────────────────────────────────────────────
# F-2 | Save label_map.json
# ─────────────────────────────────────────────────────────────────────────────

LABEL_MAP_PATH = RUN_DIR / "label_map.json"

label_map_payload = {
    # str keys for JSON compatibility
    "idx_to_label" : {str(k): v for k, v in idx_to_label.items()},
    "label_to_idx" : label_to_idx,
    "label_names"  : label_names,
    "num_classes"  : CONFIG["num_classes"],
    "backbone"     : CONFIG["backbone_name"],
    "image_size"   : list(CONFIG["image_size"]),
    "run_id"       : CONFIG["run_id"],
}

with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(label_map_payload, f, indent=2, ensure_ascii=False)

print(f"label_map.json saved → {LABEL_MAP_PATH}")
print(f"  Classes : {label_map_payload['num_classes']}")
print(f"  Backbone: {label_map_payload['backbone']}")
print(f"  Labels  : {label_names}")


label_map.json saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\label_map.json
  Classes : 6
  Backbone: EfficientNetB3
  Labels  : ['Stage1_Seedling', 'Stage2_Early_Vegetative', 'Stage3_Flowering_Initiation', 'Stage4_Flowering', 'Stage5_Unripe', 'Stage6_Ripe']


In [69]:
# ─────────────────────────────────────────────────────────────────────────────
# F-3 | Save metrics.json + classification_report.txt
# ─────────────────────────────────────────────────────────────────────────────

METRICS_PATH    = RUN_DIR / "metrics.json"
CLF_REPORT_PATH = RUN_DIR / "classification_report.txt"

# ── Extract classification report text (safe on re-runs: pop with fallback)
clf_report_text = all_metrics.pop("classification_report", None) or clf_report_text
with open(CLF_REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(clf_report_text)
print(f"Classification report saved → {CLF_REPORT_PATH}")

# ── Augment metrics dict with config snapshot before serialising
metrics_to_save = {
    **all_metrics,
    "run_id"     : CONFIG["run_id"],
    "backbone"   : CONFIG["backbone_name"],
    "image_size" : list(CONFIG["image_size"]),
    "num_classes": CONFIG["num_classes"],
    "tta"        : CONFIG["tta"],
    "tta_steps"  : CONFIG["tta_steps"],
}

# Convert any non-serialisable NumPy scalars
def _to_python(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

metrics_serialisable = {
    k: {ck: _to_python(cv) for ck, cv in v.items()} if isinstance(v, dict) else _to_python(v)
    for k, v in metrics_to_save.items()
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_serialisable, f, indent=2, ensure_ascii=False)

print(f"Metrics JSON saved → {METRICS_PATH}")


Classification report saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\classification_report.txt
Metrics JSON saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\metrics.json


In [72]:
# ─────────────────────────────────────────────────────────────────────────────
# F-4 | Final Artifact Summary
#       Prints run metadata + verifies every expected file exists on disk
# ─────────────────────────────────────────────────────────────────────────────

HISTORY_CSV_PATH   = RUN_DIR / "training_history.csv"
HISTORY_PLOT_PATH  = RUN_DIR / "training_history_plot.png"

artifact_files = [
    ("Final model (.keras)"        , FINAL_MODEL_PATH),
    ("Best checkpoint (_best.keras)", CHECKPOINT_PATH),
    ("Label map (label_map.json)"  , LABEL_MAP_PATH),
    ("Metrics (metrics.json)"      , METRICS_PATH),
    ("Classification report (.txt)", CLF_REPORT_PATH),
    ("Confusion matrix (.png)"     , CM_SAVE_PATH),
    ("Misclassified grid (.png)"   , MISCLASSIFIED_SAVE_PATH),
    ("ROC curves (.png)"           , ROC_SAVE_PATH),
    ("Training history CSV"        , HISTORY_CSV_PATH),
    ("Training history plot (.png)", HISTORY_PLOT_PATH),
]

DIVIDER = "═" * 70

print(DIVIDER)
print(f"{'TRAINING RUN COMPLETE':^70}")
print(DIVIDER)
print(f"  Run ID     : {CONFIG['run_id']}")
print(f"  Backbone   : {CONFIG['backbone_name']}")
print(f"  Image size : {CONFIG['image_size']}")
print(f"  Num classes: {CONFIG['num_classes']}")
print(f"  TTA        : {CONFIG['tta']}  (steps={CONFIG['tta_steps']})")
print(f"  Prog resize: {CONFIG['progressive_resizing']}")
print()
print(f"  Test Accuracy    : {metrics_serialisable['accuracy']:.4f}")
print(f"  F1  (macro)      : {metrics_serialisable['f1_macro']:.4f}")
print(f"  F1  (weighted)   : {metrics_serialisable['f1_weighted']:.4f}")
print(f"  ROC-AUC (macro)  : {metrics_serialisable['roc_auc_macro_ovr']:.4f}")
print()
print(f"{'ARTIFACT STATUS':^70}")
print("─" * 70)

all_present = True
for label, path in artifact_files:
    path = pathlib.Path(path)
    exists = path.exists()
    status = "✓" if exists else "✗ MISSING"
    if not exists:
        all_present = False
    print(f"  {status}  {label}")
    print(f"        → {path}")

print(DIVIDER)
if all_present:
    print(f"{'ALL ARTIFACTS SAVED SUCCESSFULLY':^70}")
else:
    print(f"{'WARNING: SOME ARTIFACTS ARE MISSING — CHECK CELLS ABOVE':^70}")
print(DIVIDER)


══════════════════════════════════════════════════════════════════════
                        TRAINING RUN COMPLETE                         
══════════════════════════════════════════════════════════════════════
  Run ID     : growth_stage_20260302_170744
  Backbone   : EfficientNetB3
  Image size : (300, 300)
  Num classes: 6
  TTA        : True  (steps=5)
  Prog resize: True

  Test Accuracy    : 0.9862
  F1  (macro)      : 0.9899
  F1  (weighted)   : 0.9862
  ROC-AUC (macro)  : 0.9990

                           ARTIFACT STATUS                            
──────────────────────────────────────────────────────────────────────
  ✓  Final model (.keras)
        → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744.keras
  ✓  Best checkpoint (_best.keras)
        → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_20260302_170744_best.keras
  ✓  Label map (label_map.json)
        → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\label_map.js

---

## Section G — Inference Function 

Implements `predict_growth_stage(image_bytes_or_path)` using the **same preprocessing and normalisation pipeline** as training.  
Accepts either a filesystem path or raw `bytes` so it can be wired to HTTP endpoints, MQTT handlers, or batch scripts without any code changes.

**Return schema**
```python
{
    "class_name" : str,            # top-1 class label
    "confidence" : float,          # top-1 softmax probability
    "probs"      : {str: float},   # full distribution over all classes
    "topk"       : [(str, float)]  # sorted list of (class, prob) len=k
}
```

In [73]:
# ─────────────────────────────────────────────────────────────────────────────
# G-1 | In-Notebook Inference Function
# ─────────────────────────────────────────────────────────────────────────────
import io


def _preprocess_image_for_inference(
    image_bytes_or_path,
    image_size: tuple[int, int],
    preprocess_fn,
) -> tf.Tensor:
    """
    Load an image from a file path or raw bytes, resize, and apply the
    backbone-specific preprocessing function.

    Args:
        image_bytes_or_path : str | Path | bytes — file path or raw image bytes.
        image_size          : (H, W) as used during training.
        preprocess_fn       : Backbone-specific Keras preprocessing function.

    Returns:
        Preprocessed float32 tensor of shape [1, H, W, 3] (batch dim included).
    """
    if isinstance(image_bytes_or_path, (str, Path)):
        raw = tf.io.read_file(str(image_bytes_or_path))
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    elif isinstance(image_bytes_or_path, (bytes, bytearray)):
        img = tf.image.decode_image(
            tf.constant(image_bytes_or_path), channels=3, expand_animations=False
        )
    else:
        raise TypeError(
            f"Expected str, Path, bytes or bytearray; got {type(image_bytes_or_path)}"
        )

    img = tf.image.resize(img, image_size)                  # [H, W, 3]
    img = tf.cast(img, tf.float32)
    img = preprocess_fn(img)                                 # backbone normalisation
    img = tf.expand_dims(img, axis=0)                        # [1, H, W, 3]
    return img


def predict_growth_stage(
    image_bytes_or_path,
    *,
    model        = None,
    label_map    : dict | None = None,
    image_size   : tuple[int, int] | None = None,
    backbone     : str | None = None,
    k            : int = 3,
    tta          : bool = False,
    tta_steps    : int = 5,
) -> dict:
    """
    Predict the tomato growth stage from a single image.

    Falls back to the notebook-level globals (``best_model``, ``idx_to_label``,
    etc.) when keyword arguments are omitted — convenient for interactive use.
    Pass explicit arguments to use this function in a standalone script.

    Args:
        image_bytes_or_path : File path (str | Path) or raw image bytes.
        model               : Loaded Keras model.  Defaults to ``best_model``.
        label_map           : Dict with keys ``idx_to_label`` (str→str) and
                              ``label_names``.  Defaults to notebook globals.
        image_size          : (H, W).  Defaults to ``CONFIG["image_size"]``.
        backbone            : Backbone name string.  Defaults to ``CONFIG["backbone"]``.
        k                   : How many top-k entries to return.
        tta                 : Whether to apply test-time augmentation.
        tta_steps           : Number of TTA passes to average.

    Returns:
        {
            "class_name" : str,
            "confidence" : float,
            "probs"      : {class_name: probability, ...},
            "topk"       : [(class_name, probability), ...],
        }
    """
    # ── Resolve defaults from notebook globals ──────────────────────────────
    _model      = model      if model      is not None else best_model
    _image_size = image_size if image_size is not None else tuple(CONFIG["image_size"])
    _backbone   = backbone   if backbone   is not None else CONFIG["backbone"]

    if label_map is not None:
        _idx_to_label = {int(k_): v for k_, v in label_map["idx_to_label"].items()}
        _label_names  = label_map["label_names"]
    else:
        _idx_to_label = idx_to_label
        _label_names  = label_names

    preprocess_fn = BACKBONE_PREPROCESS_MAP.get(_backbone, BACKBONE_PREPROCESS_MAP["EfficientNetB3"])

    # ── Preprocess ──────────────────────────────────────────────────────────
    img_tensor = _preprocess_image_for_inference(image_bytes_or_path, _image_size, preprocess_fn)

    # ── Forward pass (single or TTA) ────────────────────────────────────────
    if tta and tta_steps > 1:
        accumulated = None
        for _ in range(tta_steps):
            aug = _rotation_layer(img_tensor, training=True)
            prob_step = _model.predict(aug, verbose=0)
            accumulated = prob_step if accumulated is None else accumulated + prob_step
        probs_array = (accumulated / tta_steps)[0]
    else:
        probs_array = _model.predict(img_tensor, verbose=0)[0]

    # ── Build output dict ───────────────────────────────────────────────────
    top1_idx    = int(np.argmax(probs_array))
    class_name  = _idx_to_label[top1_idx]
    confidence  = float(probs_array[top1_idx])

    probs_dict  = {_idx_to_label[i]: float(probs_array[i]) for i in range(len(_label_names))}
    topk_sorted = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:k]

    return {
        "class_name": class_name,
        "confidence": confidence,
        "probs"     : probs_dict,
        "topk"      : topk_sorted,
    }


print("predict_growth_stage() defined — ready for Section J sanity tests.")


predict_growth_stage() defined — ready for Section J sanity tests.


---

## Section H — Save Inference Module

Writes a **standalone** Python module to  
`src/agritwin_gh/models/growth_stage_inference.py`

The module:
- Has **no dependency on notebook globals** — all state is loaded from disk.
- Caches the model after the first call (avoids re-loading on repeated predictions).
- Resolves `model_path` and `label_map_path` automatically via the naming convention if not supplied.
- Exposes a single public function: `predict_growth_stage(image_bytes_or_path, ...)`.


In [74]:
# ─────────────────────────────────────────────────────────────────────────────
# H-1 | Write src/agritwin_gh/models/growth_stage_inference.py
# ─────────────────────────────────────────────────────────────────────────────

INFERENCE_MODULE_PATH = REPO_ROOT / "src" / "agritwin_gh" / "models" / "growth_stage_inference.py"
INFERENCE_MODULE_PATH.parent.mkdir(parents=True, exist_ok=True)

_MODULE_SOURCE = '''\
"""
growth_stage_inference.py
──────────────────────────────────────────────────────────────────────────────
Standalone inference module for the Tomato Growth Stage classifier.
Trained with Task-1 of the AgriTwin-GH pipeline.

Usage
-----
>>> from agritwin_gh.models.growth_stage_inference import predict_growth_stage
>>> result = predict_growth_stage("path/to/leaf.jpg")
>>> print(result["class_name"], result["confidence"])

Or with raw bytes:
>>> with open("leaf.jpg", "rb") as f:
...     result = predict_growth_stage(f.read())

With explicit paths (e.g. for serving from a container):
>>> result = predict_growth_stage(
...     img_bytes,
...     model_path="models/growth_stage_20260301_120000.keras",
...     label_map_path="models/artifacts/growth_stage_20260301_120000/label_map.json",
... )
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Union

import numpy as np

# Lazy TensorFlow import ─ keeps module-side-effects minimal
_tf    = None
_keras = None


def _import_tf():
    global _tf, _keras
    if _tf is None:
        import tensorflow as tf   # noqa: PLC0415
        import keras              # noqa: PLC0415
        _tf    = tf
        _keras = keras
    return _tf, _keras


# ── Backbone preprocessing map (must mirror training CONFIG) ──────────────────

def _get_preprocess_fn(backbone: str):
    """Return the Keras preprocessing function for a given backbone name."""
    _import_tf()
    from keras.applications import (  # noqa: PLC0415
        efficientnet,
        efficientnet_v2,
        mobilenet_v2,
        resnet_v2,
        inception_v3,
    )
    _map = {
        "EfficientNetB0" : efficientnet.preprocess_input,
        "EfficientNetB3" : efficientnet.preprocess_input,
        "EfficientNetV2S": efficientnet_v2.preprocess_input,
        "MobileNetV2"    : mobilenet_v2.preprocess_input,
        "ResNet50V2"     : resnet_v2.preprocess_input,
        "InceptionV3"    : inception_v3.preprocess_input,
    }
    return _map.get(backbone, efficientnet.preprocess_input)


# ── Module-level model/label-map cache ───────────────────────────────────────

_CACHE: dict = {}


def _load_artifacts(model_path: Path, label_map_path: Path):
    """Load and cache the Keras model + label map (thread-safe for read-only use)."""
    cache_key = (str(model_path), str(label_map_path))
    if cache_key in _CACHE:
        return _CACHE[cache_key]

    tf, keras = _import_tf()

    model = keras.models.load_model(str(model_path))

    with open(label_map_path, "r", encoding="utf-8") as f:
        label_map = json.load(f)

    idx_to_label = {int(k): v for k, v in label_map["idx_to_label"].items()}
    label_names  = label_map["label_names"]
    backbone     = label_map.get("backbone", "EfficientNetB3")
    image_size   = tuple(label_map.get("image_size", [300, 300]))

    entry = (model, idx_to_label, label_names, backbone, image_size)
    _CACHE[cache_key] = entry
    return entry


# ── Low-level image preprocessing ────────────────────────────────────────────

def _preprocess_image(
    image_bytes_or_path: Union[str, Path, bytes, bytearray],
    image_size: tuple[int, int],
    preprocess_fn,
):
    """
    Load an image from a path or bytes, resize, normalise, and add batch dim.

    Returns:
        tf.Tensor of shape [1, H, W, 3] with dtype float32.
    """
    tf, _ = _import_tf()

    if isinstance(image_bytes_or_path, (str, Path)):
        raw = tf.io.read_file(str(image_bytes_or_path))
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    elif isinstance(image_bytes_or_path, (bytes, bytearray)):
        img = tf.image.decode_image(
            tf.constant(bytes(image_bytes_or_path)),
            channels=3,
            expand_animations=False,
        )
    else:
        raise TypeError(
            f"image_bytes_or_path must be str | Path | bytes | bytearray; "
            f"got {type(image_bytes_or_path).__name__}"
        )

    img = tf.image.resize(img, image_size)           # [H, W, 3]
    img = tf.cast(img, tf.float32)
    img = preprocess_fn(img)                          # backbone normalisation
    return tf.expand_dims(img, axis=0)                # [1, H, W, 3]


# ── Public API ────────────────────────────────────────────────────────────────

def predict_growth_stage(
    image_bytes_or_path: Union[str, Path, bytes, bytearray],
    *,
    model_path     : Union[str, Path, None] = None,
    label_map_path : Union[str, Path, None] = None,
    k              : int  = 3,
    tta            : bool = False,
    tta_steps      : int  = 5,
) -> dict:
    """
    Classify a tomato plant image into one of the six growth stages.

    Parameters
    ----------
    image_bytes_or_path
        Filesystem path (str or Path) **or** raw image bytes.
    model_path
        Path to the ``.keras`` model file.
        If omitted, the latest ``growth_stage_*.keras`` in this module's
        directory is used.
    label_map_path
        Path to ``label_map.json``.
        If omitted, inferred as ``artifacts/<run_id>/label_map.json``
        relative to this module's directory.
    k
        Number of top predictions included in the ``topk`` list.
    tta
        Enable Test-Time Augmentation (small random rotations averaged).
    tta_steps
        Number of TTA forward passes to average (only used when ``tta=True``).

    Returns
    -------
    dict
        ``class_name``  – top-1 class label (e.g. ``"Stage3_Flowering_Initiation"``)
        ``confidence``  – top-1 softmax probability in [0, 1]
        ``probs``       – full {class_name: probability} mapping over all classes
        ``topk``        – [(class_name, probability)] sorted descending, length ``k``
    """
    tf, _ = _import_tf()

    # ── Resolve artifact paths ────────────────────────────────────────────────
    HERE = Path(__file__).resolve().parent

    if model_path is None:
        candidates = sorted(HERE.glob("growth_stage_*.keras"))
        if not candidates:
            raise FileNotFoundError(
                f"No growth_stage_*.keras found in {HERE}. "
                "Supply model_path= explicitly."
            )
        model_path = candidates[-1]      # latest alphabetically == latest run

    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    if label_map_path is None:
        run_id         = model_path.stem   # e.g. growth_stage_20260301_120000
        label_map_path = HERE / "artifacts" / run_id / "label_map.json"

    label_map_path = Path(label_map_path)
    if not label_map_path.exists():
        raise FileNotFoundError(f"label_map.json not found: {label_map_path}")

    # ── Load (cached after first call) ────────────────────────────────────────
    model, idx_to_label, label_names, backbone, image_size = _load_artifacts(
        model_path, label_map_path
    )
    preprocess_fn = _get_preprocess_fn(backbone)

    # ── Preprocess image ──────────────────────────────────────────────────────
    img_tensor = _preprocess_image(image_bytes_or_path, image_size, preprocess_fn)

    # ── Inference ─────────────────────────────────────────────────────────────
    if tta and tta_steps > 1:
        import keras.layers as kl           # noqa: PLC0415
        rot_layer   = kl.RandomRotation(factor=0.08)
        accumulated = None
        for _ in range(tta_steps):
            aug      = rot_layer(img_tensor, training=True)
            step_out = model(aug, training=False).numpy()
            accumulated = step_out if accumulated is None else accumulated + step_out
        probs_array = (accumulated / tta_steps)[0]
    else:
        probs_array = model(img_tensor, training=False).numpy()[0]

    # ── Build result dict ─────────────────────────────────────────────────────
    top1_idx   = int(np.argmax(probs_array))
    class_name = idx_to_label[top1_idx]
    confidence = float(probs_array[top1_idx])

    probs_dict = {
        idx_to_label[i]: float(probs_array[i])
        for i in range(len(label_names))
    }
    topk = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:k]

    return {
        "class_name": class_name,
        "confidence": confidence,
        "probs"     : probs_dict,
        "topk"      : topk,
    }
'''

INFERENCE_MODULE_PATH.write_text(_MODULE_SOURCE, encoding="utf-8")
print(f"Inference module saved → {INFERENCE_MODULE_PATH}")
print(f"  Size: {INFERENCE_MODULE_PATH.stat().st_size:,} bytes")


Inference module saved → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_inference.py
  Size: 9,591 bytes


---

## Section I — Deployment Notes

Writes `deployment_notes.txt` into the run artifact directory:  
`src/agritwin_gh/models/artifacts/<run_id>/deployment_notes.txt`

Documents how to load the model, interpret the label map, and call the prediction function from any Python environment.


In [76]:
# ─────────────────────────────────────────────────────────────────────────────
# I-1 | Write deployment_notes.txt to the run artifact directory
# ─────────────────────────────────────────────────────────────────────────────

DEPLOYMENT_NOTES_PATH = RUN_DIR / "deployment_notes.txt"

_DEPLOYMENT_NOTES = f"""\
╔══════════════════════════════════════════════════════════════════════════════╗
║          TOMATO GROWTH STAGE CLASSIFIER — DEPLOYMENT NOTES                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

Run ID    : {CONFIG['run_id']}
Backbone  : {CONFIG['backbone_name']}
Image size: {CONFIG['image_size']}
Classes   : {CONFIG['num_classes']}
Trained   : {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M')}

──────────────────────────────────────────────────────────────────────────────
ARTIFACT LOCATIONS (relative to src/agritwin_gh/models/)
──────────────────────────────────────────────────────────────────────────────
  Final model     : {CONFIG['run_id']}.keras
  Best checkpoint : {CONFIG['run_id']}_best.keras
  Label map       : artifacts/{CONFIG['run_id']}/label_map.json
  Metrics         : artifacts/{CONFIG['run_id']}/metrics.json
  Training report : artifacts/{CONFIG['run_id']}/classification_report.txt
  Confusion matrix: artifacts/{CONFIG['run_id']}/confusion_matrix.png
  ROC curves      : artifacts/{CONFIG['run_id']}/roc_curves.png
  History CSV     : artifacts/{CONFIG['run_id']}/training_history.csv

──────────────────────────────────────────────────────────────────────────────
1. MODEL LOADING
──────────────────────────────────────────────────────────────────────────────
Using the standalone inference module (recommended):

    from agritwin_gh.models.growth_stage_inference import predict_growth_stage

    # Paths are auto-resolved — just call the function:
    result = predict_growth_stage("path/to/image.jpg")

Using Keras directly:

    import keras
    model = keras.models.load_model(
        "src/agritwin_gh/models/{CONFIG['run_id']}.keras"
    )

──────────────────────────────────────────────────────────────────────────────
2. LABEL MAP USAGE
──────────────────────────────────────────────────────────────────────────────
    import json
    with open("artifacts/{CONFIG['run_id']}/label_map.json") as f:
        label_map = json.load(f)

    # Integer index → class name
    idx_to_label = {{int(k): v for k, v in label_map["idx_to_label"].items()}}
    # e.g. {{0: "Stage1_Seedling", 1: "Stage2_Early_Vegetative", ...}}

    # Model output argmax → class name
    import numpy as np
    probs      = model.predict(img_tensor)[0]   # shape [6]
    class_name = idx_to_label[int(np.argmax(probs))]

──────────────────────────────────────────────────────────────────────────────
3. CALLING THE PREDICTION FUNCTION
──────────────────────────────────────────────────────────────────────────────
Signature:
    predict_growth_stage(
        image_bytes_or_path,   # str | Path | bytes | bytearray
        *,
        model_path=None,       # auto-resolved if None
        label_map_path=None,   # auto-resolved if None
        k=3,                   # number of top-k classes returned
        tta=False,             # enable Test-Time Augmentation
        tta_steps=5,           # TTA forward passes to average
    ) -> dict

Example — file path:
    from agritwin_gh.models.growth_stage_inference import predict_growth_stage

    result = predict_growth_stage("data/external/Tomato Growth Stages/Stage3_Flowering_Initiation/img001.jpg")
    print(result["class_name"])    # "Stage3_Flowering_Initiation"
    print(result["confidence"])    # 0.9213
    print(result["topk"])
    # [("Stage3_Flowering_Initiation", 0.9213),
    #  ("Stage4_Flowering",            0.0612),
    #  ("Stage2_Early_Vegetative",     0.0103)]

Example — raw bytes (e.g. from HTTP upload or MQTT payload):
    with open("leaf.jpg", "rb") as fh:
        result = predict_growth_stage(fh.read())

Example — with TTA for higher-confidence edge cases:
    result = predict_growth_stage(img_path, tta=True, tta_steps=8)

──────────────────────────────────────────────────────────────────────────────
4. CLASS LABELS (ordered by integer index)
──────────────────────────────────────────────────────────────────────────────
  0 → Stage1_Seedling
  1 → Stage2_Early_Vegetative
  2 → Stage3_Flowering_Initiation
  3 → Stage4_Flowering
  4 → Stage5_Unripe
  5 → Stage6_Ripe

──────────────────────────────────────────────────────────────────────────────
5. PREPROCESSING CONTRACT (must match training, handled automatically by
   growth_stage_inference.py)
──────────────────────────────────────────────────────────────────────────────
  • Decode image → RGB
  • Resize to {CONFIG['image_size']} using bilinear interpolation
  • Cast to float32
  • Apply keras.applications.efficientnet.preprocess_input (scales to [-1, 1])
  • Expand dims to [1, H, W, 3] for model input

──────────────────────────────────────────────────────────────────────────────
6. RUNTIME DEPENDENCIES
──────────────────────────────────────────────────────────────────────────────
  tensorflow >= 2.13
  keras      >= 2.13
  numpy      >= 1.24
  Pillow     (optional, for PIL-based loading in custom pipelines)

Install with:
  uv pip install tensorflow keras numpy pillow

──────────────────────────────────────────────────────────────────────────────
END OF DEPLOYMENT NOTES
──────────────────────────────────────────────────────────────────────────────
"""

DEPLOYMENT_NOTES_PATH.write_text(_DEPLOYMENT_NOTES, encoding="utf-8")
print(f"Deployment notes saved → {DEPLOYMENT_NOTES_PATH}")


Deployment notes saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\deployment_notes.txt


---

## Section J — Package Check & Sanity Tests 

Validates the full inference pipeline **end-to-end from cold start** by loading
the saved checkpoint and label map from disk (not from notebook memory), then
running predictions via both input modes.

| Test | Expected outcome |
|---|---|
| Load saved model + `label_map.json` | No errors |
| `predict_growth_stage(file_path)` | Returns valid dict |
| `predict_growth_stage(raw_bytes)` | Same top-1 as path prediction |
| Top-3 probability sum ≤ 1 | Sanity check on softmax |
| TTA comparison (if enabled) | Shows single-pass vs TTA probabilities |


In [77]:
# ─────────────────────────────────────────────────────────────────────────────
# J-1 | Package Check (uv-first)
#       No new dependencies are required; all packages used here (tensorflow,
#       keras, numpy, Pillow) are already part of the training environment.
#       This cell is a guard that verifies importability and logs versions.
# ─────────────────────────────────────────────────────────────────────────────
import importlib
import subprocess
import sys

_REQUIRED = {
    "tensorflow": "tensorflow",
    "keras"     : "keras",
    "numpy"     : "numpy",
    "PIL"       : "pillow",   # import name → install name
}

_missing = []
for import_name, install_name in _REQUIRED.items():
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "n/a")
        print(f"  ✓  {import_name:<15} {ver}")
    except ModuleNotFoundError:
        _missing.append(install_name)
        print(f"  ✗  {import_name:<15} NOT FOUND — will install")

if _missing:
    print(f"\nInstalling missing packages via uv: {_missing}")
    _pkg_str = " ".join(_missing)

    # 1. uv CLI
    ret = subprocess.run(
        ["uv", "pip", "install", *_missing],
        capture_output=True, text=True,
    )
    if ret.returncode != 0:
        # 2. uv module
        ret = subprocess.run(
            [sys.executable, "-m", "uv", "pip", "install", *_missing],
            capture_output=True, text=True,
        )
    if ret.returncode != 0:
        # 3. pip fallback
        subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

    print("Packages installed successfully.")
else:
    print("\nAll required packages present — no installation needed.")


  ✓  tensorflow      2.20.0
  ✓  keras           3.13.2
  ✓  numpy           2.4.2
  ✓  PIL             12.1.1

All required packages present — no installation needed.


In [81]:
# ─────────────────────────────────────────────────────────────────────────────
# J-2 | End-to-End Sanity Tests
#
#   a) Load saved model + label_map.json afresh from disk
#   b) Pick 1 random test image
#   c) Predict via file path
#   d) Predict via raw bytes
#   e) Assert same top-1 between (c) and (d)
#   f) Print top-3 probabilities
#   g) If TTA enabled → single-pass vs TTA comparison
# ─────────────────────────────────────────────────────────────────────────────
import random
import importlib
import sys

PASS = "✓ PASS"
FAIL = "✗ FAIL"

# ── Import the inference module written to disk in Section H ─────────────────
_models_dir = str(MODELS_DIR)
if _models_dir not in sys.path:
    sys.path.insert(0, _models_dir)

import growth_stage_inference as _gsi
importlib.reload(_gsi)
predict_growth_stage = _gsi.predict_growth_stage
print(f"predict_growth_stage imported from: {_gsi.__file__}")

# ── a) Cold-load model + label_map from disk ─────────────────────────────────
print("\nLoading model and label_map from disk (cold start) …")
sanity_model = keras.models.load_model(str(FINAL_MODEL_PATH))

with open(str(LABEL_MAP_PATH), "r", encoding="utf-8") as _f:
    _raw_lm = json.load(_f)

sanity_idx_to_label = {int(k): v for k, v in _raw_lm["idx_to_label"].items()}

print(f"  Model     : {FINAL_MODEL_PATH.name}")
print(f"  Label map : {LABEL_MAP_PATH}")
print(f"  Classes   : {_raw_lm['label_names']}")

# ── b) Pick a random test image ──────────────────────────────────────────────
# test_pairs stores (path_str, label_str) — use the label string directly
rng = random.Random(42)
sample_path_str, sample_true_label = rng.choice(test_pairs)
sample_path = pathlib.Path(sample_path_str)

print(f"\nSample image : {sample_path.name}")
print(f"True label   : {sample_true_label}")

# ── c) Predict via file path ─────────────────────────────────────────────────
result_path = predict_growth_stage(
    sample_path,
    model_path     = FINAL_MODEL_PATH,
    label_map_path = LABEL_MAP_PATH,
    k              = 3,
    tta            = False,
)

print("\n" + "─" * 56)
print("Prediction via FILE PATH (single-pass)")
print("─" * 56)
print(f"  Top-1 class  : {result_path['class_name']}")
print(f"  Confidence   : {result_path['confidence']:.4f}  ({result_path['confidence']:.1%})")
print("  Top-3 classes:")
for rank, (cls, prob) in enumerate(result_path["topk"], start=1):
    bar = "█" * int(prob * 30)
    print(f"    {rank}. {cls:<40s} {prob:.4f}  {bar}")

# ── d) Predict via raw bytes ─────────────────────────────────────────────────
with open(sample_path, "rb") as _fh:
    sample_bytes = _fh.read()

result_bytes = predict_growth_stage(
    sample_bytes,
    model_path     = FINAL_MODEL_PATH,
    label_map_path = LABEL_MAP_PATH,
    k              = 3,
    tta            = False,
)

print("\n" + "─" * 56)
print("Prediction via RAW BYTES (single-pass)")
print("─" * 56)
print(f"  Top-1 class  : {result_bytes['class_name']}")
print(f"  Confidence   : {result_bytes['confidence']:.4f}  ({result_bytes['confidence']:.1%})")

# ── e) Assert top-1 match ────────────────────────────────────────────────────
top1_match   = result_path["class_name"] == result_bytes["class_name"]
top1_status  = PASS if top1_match else FAIL
conf_match   = abs(result_path["confidence"] - result_bytes["confidence"]) < 1e-5
conf_status  = PASS if conf_match else FAIL

print("\n" + "─" * 56)
print("Consistency Check: path vs bytes")
print("─" * 56)
print(f"  Top-1 identical   : {top1_status}  "
      f"({result_path['class_name']} == {result_bytes['class_name']})")
print(f"  Confidence match  : {conf_status}  "
      f"({result_path['confidence']:.6f} vs {result_bytes['confidence']:.6f})")

# ── f) Softmax sanity (probs sum to ~1) ──────────────────────────────────────
prob_sum = sum(result_path["probs"].values())
prob_ok  = abs(prob_sum - 1.0) < 1e-4
print(f"  Prob-sum ≈ 1.000  : {PASS if prob_ok else FAIL}  ({prob_sum:.6f})")

# Ground-truth agreement
gt_ok = result_path["class_name"] == sample_true_label
print(f"  Ground-truth match: {PASS if gt_ok else '— predicted different class (not a bug)'}")

# ── g) TTA comparison (optional) ─────────────────────────────────────────────
if CONFIG["tta"]:
    print("\n" + "─" * 56)
    print(f"TTA Comparison  (tta_steps={CONFIG['tta_steps']})")
    print("─" * 56)

    result_tta = predict_growth_stage(
        sample_path,
        model_path     = FINAL_MODEL_PATH,
        label_map_path = LABEL_MAP_PATH,
        k              = 3,
        tta            = True,
        tta_steps      = CONFIG["tta_steps"],
    )

    print(f"  {'Class':<40} {'Single-pass':>12} {'TTA':>8}")
    print("  " + "─" * 60)
    for cls in result_path["probs"]:
        sp_prob  = result_path["probs"][cls]
        tta_prob = result_tta["probs"][cls]
        delta    = tta_prob - sp_prob
        arrow    = "↑" if delta > 0.001 else ("↓" if delta < -0.001 else "≈")
        print(f"  {cls:<40} {sp_prob:>12.4f} {tta_prob:>8.4f} {arrow}")

    print(f"\n  Single-pass top-1 : {result_path['class_name']}  ({result_path['confidence']:.4f})")
    print(f"  TTA top-1         : {result_tta['class_name']}  ({result_tta['confidence']:.4f})")
    tta_consistent = result_path["class_name"] == result_tta["class_name"]
    print(f"  Prediction agrees : {PASS if tta_consistent else '⚠  different top-1 (TTA shifted confidence)'}")
else:
    print("\n[TTA not enabled in CONFIG — skipping TTA comparison]")
    print("Set CONFIG['tta'] = True and re-run to see TTA vs single-pass comparison.")

print("\n" + "═" * 56)
print("SANITY TESTS COMPLETE")
print("═" * 56)


predict_growth_stage imported from: E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_inference.py

Loading model and label_map from disk (cold start) …
  Model     : growth_stage_20260302_170744.keras
  Label map : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_20260302_170744\label_map.json
  Classes   : ['Stage1_Seedling', 'Stage2_Early_Vegetative', 'Stage3_Flowering_Initiation', 'Stage4_Flowering', 'Stage5_Unripe', 'Stage6_Ripe']

Sample image : 90_IMG_20095108_jpg.rf.0b9c74cb620efff47f00d4480b0ce56b.jpg
True label   : Stage4_Flowering

────────────────────────────────────────────────────────
Prediction via FILE PATH (single-pass)
────────────────────────────────────────────────────────
  Top-1 class  : Stage4_Flowering
  Confidence   : 0.9607  (96.1%)
  Top-3 classes:
    1. Stage4_Flowering                         0.9607  ████████████████████████████
    2. Stage6_Ripe                              0.0166  
    3. Stage1_Seedling                          0.0082  

─